# 서울 폐의약품 배출 이동 부담과 편의점 보완 거점 분석

**분석 지역:** 봉천동 일대 9개 행정동, 창신·숭인동 5개 행정동.

기존 수거망을 유지하면서 주민의 왕복 보행거리와 정류소 경유 추가거리를 줄이는 편의점을 탐색
이어 전체 주민의 개선을 일정 수준 유지하는 고령층 우선안을 비교합니다.

- 데이터 시점은 인구 2024년
- 행정동·집계구 경계 2025년 2분기, 상가 2026년 6월
- 정류소 2026년 9월
- 수거망·보행망 조회 2026년 10월 4일입니다.


## 0. 환경과 실행 설정

- `REBUILD_INPUTS=True`이면 별도로 준비한 원자료에서 전처리를 다시 수행합
- `False`이면 별도로 준비한 정제자료를 사용, 두 경우 모두 기본·확장 최적화는 다시 계산
- 검증된 거리 행렬 캐시는 입력과 코드의 해시가 같을 때만 재사용됨, 완전히 새로 경로를 계산하려면 `REBUILD_ROUTES=True`로 설정

- 설치가 필요한 경우에만 `INSTALL_MISSING_PACKAGES=True`로 수정
- 첫 실행에는 전처리·보행망·최적화 계산 시간이 필요

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, importlib, importlib.util, subprocess, time
from contextlib import redirect_stdout

INSTALL_MISSING_PACKAGES = False
REBUILD_INPUTS = True
REBUILD_ROUTES = False

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/raw/expanded_osm/manifest.json').exists():
    raise FileNotFoundError('노트북을 압축 해제한 폴더에서 실행하세요. data 폴더가 같은 위치에 있어야 합니다.')
for folder in ['src', 'results/resident_analysis', 'results/notebook', '.cache/matplotlib']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
sys.path.insert(0, str(ROOT / 'src'))
required = ['numpy','pandas','scipy','geopandas','shapely','pyproj','networkx','matplotlib','openpyxl','pyogrio']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError('필요한 패키지: ' + ', '.join(missing) + '. 이 셀의 INSTALL_MISSING_PACKAGES를 True로 바꾸고 실행하세요.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')])
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display, Image, Markdown
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

def run_stage(label, callback):
    started = time.monotonic()
    with (ROOT / 'results/notebook' / (label + '.log')).open('w', encoding='utf-8') as log:
        with redirect_stdout(log):
            value = callback()
    print(f'{label}: 완료 ({time.monotonic() - started:.1f}초)')
    return value

print('실행 환경 준비 완료. 자료 경로는 현재 폴더 기준입니다.')

## 1. 원자료 확인

In [ ]:
records = read_json('reference/input_checksums.json')
osm_manifest = read_json('data/raw/expanded_osm/manifest.json')
records += [{'file': r['file'], 'sha256': r['sha256']} for r in osm_manifest['files']]
checks = []
for record in records:
    path = ROOT / record['file']
    if not path.is_file():
        raise FileNotFoundError(record['file'])
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checks.append({'파일':record['file'], '용량_MB':path.stat().st_size / 1024**2,
                   '원본과_동일':digest == record['sha256']})
assert all(row['원본과_동일'] for row in checks), '원본 해시가 다른 파일이 있습니다.'
print(f'원자료 및 도로망 {len(checks)}개 파일의 해시가 일치합니다.')
display(pd.DataFrame(checks[:9]).round(2))

## 2. 기존 수거망·편의점·정류소 정제

In [ ]:
%%writefile src/extract_convenience.py
"""Extract convenience stores from the exact SEMAS source CSV (G20405)."""
import csv
import json
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]

def main():
    source = ROOT / 'data/raw/seoul_stores_202606.csv'
    output = ROOT / 'data/raw/seoul_convenience_202606.json'
    with source.open(encoding='utf-8-sig', newline='') as f:
        rows = [r for r in csv.DictReader(f) if r['상권업종소분류코드'] == 'G20405']
    if not rows:
        raise ValueError('편의점 코드 G20405에 해당하는 기록이 없습니다.')
    output.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Convenience records: {len(rows)}')

if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/prepare_inputs.py
"""Normalize public source records, retaining provenance and candidate flags."""
import json, hashlib, collections, re
from pathlib import Path
import pandas as pd

ROOT=Path(__file__).resolve().parents[1]
RAW=ROOT/'data/raw'; OUT=ROOT/'data/processed'
OUT.mkdir(exist_ok=True, parents=True)
(ROOT/'results').mkdir(parents=True,exist_ok=True)
def write(name,obj):
    (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
def district(addr):
    m=re.search(r'(\S+구)\s',addr+' ')
    return m.group(1) if m else None

fs=json.loads((RAW/'seoul_smartmap_bins_20261004.json').read_text())['body']['geojsons']['1649132420936']['features']
bins=[]
for f in fs:
    p=f['properties']; lon,lat=f['geometry']['coordinates'][:2]
    addr=p.get('cot_addr_full_new') or p.get('cot_addr_full_old') or ''
    if not (126 < lon < 128 and 37 < lat < 38): continue
    bins.append(dict(id='BIN_'+p['cot_conts_id'],name=p.get('cot_value_01') or p['cot_conts_name'],
                     lon=lon,lat=lat,address=addr,district=district(addr),kind='전용 수거함',
                     category=p.get('sub_cate_name'),source='스마트서울맵',checked_on='2026-10-04',
                     opening_hours=None,medicine_types='공식 안내상 물약 포함',site_operation_verified=False))
write('existing_bins.json',bins)
rawpost=json.loads((RAW/'seoul_postboxes_search_20261004.json').read_text())
postgroups=collections.defaultdict(list)
for r in rawpost: postgroups[r['postId']].append(r)
posts=[]; conflicting=[]
for key,group in postgroups.items():
    coordinate_pairs={(r['postLat'],r['postLon']) for r in group}
    if len(coordinate_pairs)>1: conflicting.append(key)
    r=max(group,key=lambda r:r.get('modDt','')); lon=float(r['postLon']); lat=float(r['postLat'])
    if not (126 < lon < 128 and 37 < lat < 38): continue
    posts.append(dict(id='POST_'+key,name='우체통',lon=lon,lat=lat,address=r['postAddr'],district=district(r['postAddr']),
                      kind='우체통',source='인터넷우체국',checked_on='2026-10-04',source_modified=r.get('modDt'),
                      collection_time=r.get('postServiceTime'),medicine_types='알약·가루약 등; 물약 제외',
                      medicine_acceptance_basis='서울시 우체통 수거 정책',site_operation_verified=False,duplicate_source_rows=len(group)))
write('postboxes.json',posts)

stores=json.loads((RAW/'seoul_convenience_202606.json').read_text())
groups=collections.defaultdict(list)
for r in stores:
    try: lon,lat=float(r['경도']),float(r['위도'])
    except (ValueError,TypeError): continue
    if 126 < lon < 128 and 37 < lat < 38: groups[(lon,lat)].append(r)
candidates=[]
for (lon,lat),group in groups.items():
    names=sorted({(r['상호명']+' '+r['지점명']).strip() for r in group})
    addr=group[0]['도로명주소']; text=' '.join(names+[r['건물명'] for r in group])
    flags=[]
    if len(group)>1: flags.append('동일 좌표 복수 기록')
    if re.search('노브랜드|더프레시|THE FRESH',text,re.I): flags.append('슈퍼마켓 가능성')
    if re.search('병원|의료원|대학교|캠퍼스|부대|군부대|역내|구내',text): flags.append('시설 내부 출입 확인')
    if any(r['층정보'] and r['층정보'] not in ('1','1층') for r in group): flags.append('층 정보 확인')
    if any(r['표준산업분류코드']!='G47122' for r in group): flags.append('업종 코드 교차 확인')
    candidates.append(dict(id='CVS_'+hashlib.sha256('|'.join(sorted(r['상가업소번호'] for r in group)).encode()).hexdigest()[:12],
                           name=' / '.join(names),names=names,lon=lon,lat=lat,address=addr,district=group[0]['시군구명'],
                           dong=group[0]['행정동명'],legal_dong=group[0]['법정동명'],raw_rows=len(group),
                           source_ids=[r['상가업소번호'] for r in group],flags=flags,source_date='2026-06-30',
                           participation_verified=False,opening_hours=None))
write('convenience_candidates.json',candidates)
bus=pd.read_excel(RAW/'seoul_bus_20260902.xlsx',dtype={'NODE_ID':str,'ARS_ID':str})
busrows=[]
for _,r in bus.iterrows():
    busrows.append(dict(id=r['NODE_ID'],ars_id=str(r['ARS_ID']).zfill(5),name=r['정류소명'],lon=float(r['X좌표']),lat=float(r['Y좌표']),kind=r['정류소타입']))
write('bus_stops.json',busrows)

districts=[]
for gu in ['관악구','종로구']:
    cc=[r for r in candidates if r['district']==gu]
    districts.append(dict(district=gu,convenience_raw=sum(r['시군구명']==gu for r in stores),convenience_locations=len(cc),
                          candidates_with_flags=sum(bool(r['flags']) for r in cc),dedicated_bins=sum(r['district']==gu for r in bins),
                          postboxes=sum(r['district']==gu for r in posts)))
audit=dict(checked_on='2026-10-04',districts=districts,seoul_convenience_records=len(stores),seoul_convenience_locations=len(candidates),
           smartmap_feature_count=len(fs),smartmap_valid_coordinate_count=len(bins),postbox_raw_count=len(rawpost),postbox_unique_ids=len(postgroups),
           postbox_coordinate_conflicts=conflicting,bus_records=len(busrows),
           caveats=['수거함과 우체통은 기관별 원천 명부이며 실제 수거 여부는 개별 확인 전.',
                    '전용 수거함 명부의 구별 건수는 주소에서 구를 확인할 수 있는 기록 기준.',
                    '편의점 위치는 업종 분류 기반 후보. 중복 좌표 정리 후 출입·업종 확인 대상 표시.',
                    '우체통 주소 검색은 서울/서울특별시 표기에 따라 일부 기록만 반환. 서울 전체 응답에서 구명을 추출함.',
                    '이 감사는 시설 정제 단계이며 주민 인구 연결은 prepare_resident_inputs.py에서 수행.'])
(ROOT/'results/data_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2))

sources=[
    ('seoul_stores_202606.csv','소상공인시장진흥공단 상가정보','https://www.data.go.kr/data/15083033/fileData.do','2026-06-30','2026-09-28에 확보한 전국 원본 ZIP에서 서울 추출'),
    ('seoul_smartmap_bins_20261004.json','스마트서울맵 폐의약품 전용수거함','https://map.seoul.go.kr/smgis2/','2026-10-04 조회','공개 지도 테마 1649132420936 getContentAll 응답'),
    ('seoul_postboxes_search_20261004.json','인터넷우체국 우체통 검색','https://m.epost.go.kr/mobile/postFind/postMapSearch.jsp','2026-10-04 조회','공개 지역 검색 서울 전체 응답. 동일 postId 중복 제거'),
    ('seoul_bus_20260902.xlsx','서울시 버스정류소 위치정보','https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do','2026-09-02','X/Y 값으로 WGS84 경위도 확인; ARS ID 다섯 자리 문자열 처리'),
]
manifest=[]
for file,name,url,date,note in sources:
    path=RAW/file
    manifest.append(dict(name=name,path=str(path.relative_to(ROOT)),source_url=url,reference_date=date,note=note,sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
for path in RAW.glob('*osm_20261004.metadata.json'):
    manifest.append(json.loads(path.read_text()))
write('source_manifest.json',manifest)
print(json.dumps(audit,ensure_ascii=False,indent=2))


In [ ]:
import runpy
if REBUILD_INPUTS:
    run_stage('01_편의점_추출', lambda: runpy.run_path(str(ROOT/'src/extract_convenience.py'), run_name='__main__'))
    run_stage('02_시설_정제', lambda: runpy.run_path(str(ROOT/'src/prepare_inputs.py'), run_name='__main__'))
facility_files = {'전용 수거함':'existing_bins.json', '우체통':'postboxes.json',
                  '편의점 후보':'convenience_candidates.json', '정류소':'bus_stops.json'}
display(pd.DataFrame([{'자료':name, '서울_정제기록수':len(read_json('data/processed/'+filename))}
                      for name, filename in facility_files.items()]))
print('위 건수는 입력 명부 전체이며, 최적화에서는 연구 지역과 주변 경로 범위로 다시 제한합니다.')

## 3. 주민 인구와 고령인구 연결

전체 주민은 **100m 격자 중심점**에, 고령인구는 **집계구 내부 대표점**에 연결

`representative_point()`는 집계구의 경계 모양을 바탕으로 내부의 한 점을 선택
집계구의 고령인구가 이 점에서 출발한다고 가정

연령별 비공개 값은 0~4명 범위와 총인구를 함께 사용해 하한·상한을 구합니다.
기본 고령층 분석은 하한 인구를 가중치로 사용하고 상한 인구 적용 결과도 비교

In [ ]:
%%writefile src/prepare_resident_inputs.py
"""Import SGIS downloads; retain released counts and original spatial units.

The 100m grids represent total resident population. Older residents are kept
at their own census-area representative points, never uniformly allocated to
100m grids. Missing census values remain missing, including suppressed counts.
"""
import argparse, hashlib, json, unicodedata, zipfile
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
from pyproj import CRS, Transformer

ROOT=Path(__file__).resolve().parents[1]
OUT=ROOT/'data/processed/residents'
ANALYSIS_CRS=CRS.from_epsg(5179)
STUDIES={
 'bongcheon':{'district_prefix':'11210','names':['보라매동','청림동','성현동','행운동','낙성대동','청룡동','은천동','중앙동','인헌동']},
 'changsin_sungin':{'district_prefix':'11010','names':['창신1동','창신2동','창신3동','숭인1동','숭인2동']},
}

def unpack(path, folder):
    folder.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        if z.testzip() is not None: raise ValueError(f'Corrupt archive: {path.name}')
        for info in z.infolist():
            name=info.filename
            if not info.flag_bits&0x800:
                try: name=name.encode('cp437').decode('cp949')
                except UnicodeError: pass
            name=unicodedata.normalize('NFC',name)
            destination=folder/name
            if not destination.resolve().is_relative_to(folder.resolve()): raise ValueError('Unsafe archive path')
            if info.is_dir(): destination.mkdir(parents=True,exist_ok=True);continue
            destination.parent.mkdir(parents=True,exist_ok=True)
            if not destination.exists(): destination.write_bytes(z.read(info))
    for nested in list(folder.rglob('*.zip')):
        unpack(nested,nested.with_suffix(''))

def read_table(path):
    for encoding in ('utf-8-sig','cp949'):
        try:
            data=pd.read_csv(path,header=None,names=['year','id','code','value'],dtype=str,encoding=encoding)
            break
        except UnicodeDecodeError: pass
    else: raise ValueError(f'Unknown table encoding: {path}')
    data['value']=pd.to_numeric(data['value'],errors='coerce')
    return data

def read_projected(path, bounds=None):
    """Read a layer in EPSG:5179; bounds, when supplied, are also EPSG:5179.

    A bounding-box tuple must use the source layer's CRS when passed to
    Pyogrio. Convert that filter first, then reproject the returned geometry.
    Never relabel source coordinates with set_crs(..., allow_override=True).
    """
    source_crs=pyogrio.read_info(path).get('crs')
    if not source_crs:
        raise ValueError(f'{Path(path).name}: 원본 좌표계 정보가 없습니다. 함께 제공된 .prj 파일을 확인하세요.')
    source_crs=CRS.from_user_input(source_crs)
    options={'engine':'pyogrio'}
    if bounds is not None:
        options['bbox']=Transformer.from_crs(
            ANALYSIS_CRS,source_crs,always_xy=True
        ).transform_bounds(*bounds,densify_pts=21)
    frame=gpd.read_file(path,**options)
    if frame.crs is None:
        raise ValueError(f'{Path(path).name}: 읽은 도형의 좌표계가 없습니다.')
    return frame.to_crs(ANALYSIS_CRS)

def points_json(frame, count_column, prefix, source_kind):
    points=frame.copy()
    points.geometry=points.geometry.representative_point() if source_kind=='census' else points.geometry.centroid
    ll=points.to_crs(4326)
    rows=[]
    for (_,row),p in zip(points.iterrows(),ll.geometry):
        value=row[count_column]
        record=dict(id=prefix+str(row['id']),source_id=str(row['id']),lon=float(p.x),lat=float(p.y),
                         population=None if pd.isna(value) else float(value),adm_cd=str(row['adm_cd']),
                         dong=row['ADM_NM'],source_kind=source_kind,reference_year=2024)
        if source_kind=='census':
            record.update(population_lower=float(row['elderly_lower']),population_upper=float(row['elderly_upper']),
                          known_age_bins=int(row['known_age_bins']),weight_basis='연령구간·총인구에서 산출한 65세 이상 인구 하한')
        rows.append(record)
    return rows

def main(input_paths):
    (ROOT/'results').mkdir(parents=True,exist_ok=True)
    OUT.mkdir(parents=True,exist_ok=True); cache=ROOT/'.cache/sgis_seoul';cache.mkdir(parents=True,exist_ok=True)
    manifest=[]
    for path in input_paths:
        path=path.resolve()
        if not path.is_file(): raise FileNotFoundError(path)
        digest=hashlib.sha256(path.read_bytes()).hexdigest()
        unpack(path,cache/path.stem)
        manifest.append({'file':path.name,'sha256':digest})
    shapes=list(cache.rglob('*.shp'))
    admin_path=next((p for p in shapes if p.stem.startswith('bnd_dong_11_')),None)
    census_path=next((p for p in shapes if p.stem.startswith('bnd_oa_11_')),None)
    grid_paths=[p for p in shapes if p.stem.upper().endswith('_100M')]
    if not admin_path or not grid_paths: raise ValueError('서울 읍면동 경계와 100M 격자 경계가 필요합니다.')
    admins=read_projected(admin_path)
    admins['ADM_CD']=admins['ADM_CD'].astype(str)
    tables=[]
    for p in cache.rglob('*.csv'):
        if '인구' in p.name or p.name.startswith('in_') or p.name.startswith('to_in'):
            tables.append((p.name,read_table(p)))
    grid_total=pd.concat([t for name,t in tables if '_100M' in name.upper()],ignore_index=True)
    grid_total=grid_total.loc[grid_total.code=='to_in_001',['id','value']].drop_duplicates()
    if grid_total.id.duplicated().any(): raise ValueError('Conflicting population counts for one grid.')
    census_tables=[t for name,t in tables if '_100M' not in name.upper() and '_500M' not in name.upper() and '_1K' not in name.upper()]
    ct=pd.concat(census_tables,ignore_index=True).drop_duplicates() if census_tables else None
    audit={'statistics_year':2024,'boundary_year':2025,'inputs':manifest,'areas':{}}
    for area,config in STUDIES.items():
        adm=admins.loc[admins.ADM_CD.str.startswith(config['district_prefix']) & admins.ADM_NM.isin(config['names'])].copy()
        if len(adm)!=len(config['names']): raise ValueError(f'{area}: requested administrative boundaries not matched: {adm.ADM_NM.tolist()}')
        polygon=adm.geometry.union_all()
        grid_parts=[read_projected(p,bounds=polygon.bounds) for p in grid_paths]
        grids=gpd.GeoDataFrame(pd.concat(grid_parts,ignore_index=True),geometry='geometry')
        grids=grids.rename(columns={'GRID_CD':'id'}).drop_duplicates('id')
        centers=gpd.GeoDataFrame({'id':grids.id},geometry=grids.centroid,crs=grids.crs)
        mapped=gpd.sjoin(centers,adm[['ADM_CD','ADM_NM','geometry']],predicate='within',how='inner')
        if mapped.id.duplicated().any(): raise ValueError('Grid center belongs to multiple administrative polygons.')
        grids=grids.merge(mapped[['id','ADM_CD','ADM_NM']],on='id').rename(columns={'ADM_CD':'adm_cd'})
        grids=grids.merge(grid_total.rename(columns={'value':'population'}),on='id',how='left')
        residents=points_json(grids,'population','POP_','grid100m')
        (OUT/f'{area}_population.json').write_text(json.dumps(residents,ensure_ascii=False,indent=2,allow_nan=False))
        adm.to_file(OUT/'residents.gpkg',layer=f'{area}_admin',driver='GPKG')
        grids.to_file(OUT/'residents.gpkg',layer=f'{area}_grid100m',driver='GPKG')
        a={'administrative_names':adm.ADM_NM.tolist(),'grid_count':len(grids),
           'grid_known_count':int(grids.population.notna().sum()),'grid_missing_count':int(grids.population.isna().sum()),
           'released_population_sum':float(grids.population.sum()),
           'selection_rule':'100m 격자 중심이 연구 대상 행정동 내부에 있는 격자. 경계 격자의 면적 배분 없음.',
           'disclosure_note':'SGIS 격자 값에는 iLBA 비밀보호 조정이 적용됨. 원래 주민 수의 정밀 관측값으로 해석하지 않음.'}
        if census_path and ct is not None:
            census=read_projected(census_path,bounds=polygon.bounds)
            code_col=next(c for c in census.columns if c.upper() in ('TOT_OA_CD','OA_CD'))
            census=census.rename(columns={code_col:'id'});census['id']=census.id.astype(str)
            # Use the provided eight-character dong key, rather than a MOIS key.
            census['adm_cd']=census['ADM_CD'].astype(str)
            census=census.drop(columns='ADM_CD')
            census=census.merge(adm[['ADM_CD','ADM_NM']].rename(columns={'ADM_CD':'adm_cd'}),on='adm_cd')
            age_codes=[f'in_age_{i:03d}' for i in range(1,22)]
            age_data=ct.loc[ct.code.isin(age_codes),['id','code','value']].drop_duplicates()
            if age_data.duplicated(['id','code']).any(): raise ValueError('Conflicting age rows.')
            age=age_data.pivot(index='id',columns='code',values='value').reindex(index=census.id,columns=age_codes)
            totals=ct.loc[ct.code=='to_in_001',['id','value']].drop_duplicates().set_index('id').value.reindex(census.id)
            old=age[[f'in_age_{i:03d}' for i in range(14,22)]]
            young=age[[f'in_age_{i:03d}' for i in range(1,14)]]
            lower=old.sum(axis=1,min_count=0)
            # Suppressed or omitted cells are integer values from 0 to 4 under
            # SGIS's census-area rule. The total and other age bins tighten bounds.
            lower=np.maximum(lower,totals-(young.sum(axis=1)+young.isna().sum(axis=1)*4))
            upper=np.minimum(old.sum(axis=1)+old.isna().sum(axis=1)*4,totals-young.sum(axis=1))
            if totals.isna().any(): raise ValueError('Missing census total: cannot bound suppressed age cells.')
            if (upper<lower).any(): raise ValueError('Inconsistent total/age counts.')
            census['elderly_lower']=lower.to_numpy();census['elderly_upper']=upper.to_numpy()
            census['elderly']=census['elderly_lower'];census['known_age_bins']=old.notna().sum(axis=1).to_numpy()
            rows=points_json(census,'elderly','AGE_','census')
            (OUT/f'{area}_elderly.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2,allow_nan=False))
            census.to_file(OUT/'residents.gpkg',layer=f'{area}_elderly',driver='GPKG')
            a['elderly']={'census_areas':len(census),'all_eight_age_bins_published':int((census.known_age_bins==8).sum()),
                          'lower_sum':float(census.elderly_lower.sum()),'upper_sum':float(census.elderly_upper.sum()),
                          'uncertainty_rule':'65세 이상 8개 연령구간과 총인구로 하한·상한 계산. 비공개 연령구간은 0~4명.',
                          'location':'집계구 내부 대표점. 고령인구를 100m 격자로 임의 배분하지 않음.'}
        audit['areas'][area]=a
    (ROOT/'results/resident_input_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2))
    print(json.dumps(audit,ensure_ascii=False,indent=2))

if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('archives',nargs='+',type=Path)
    main(parser.parse_args().archives)


In [ ]:
from prepare_resident_inputs import main as prepare_residents
sgis_files = ['sgis_seoul_population_2024.zip', 'bnd_dong_11_2025_2Q.zip',
              'bnd_oa_11_2025_2Q.zip', 'grid_다사.zip']
if REBUILD_INPUTS:
    run_stage('03_SGIS_연결', lambda: prepare_residents([ROOT/'data/raw'/name for name in sgis_files]))
population_audit = read_json('results/resident_input_audit.json')
area_names = {'bongcheon':'봉천동 일대', 'changsin_sungin':'창신·숭인동'}
display(pd.DataFrame([{
    '지역':area_names[area], '행정동_수':len(row['administrative_names']),
    '공개_격자인구':row['released_population_sum'], '고령층_집계구_수':row['elderly']['census_areas'],
    '65세이상_인구하한':row['elderly']['lower_sum'], '65세이상_인구상한':row['elderly']['upper_sum']
} for area, row in population_audit['areas'].items()]))
print('연결 전 자료 집계, 실제 거리 비교 표본은 도로 연결·지도 경계를 점검한 뒤 확정')

In [ ]:
%%writefile src/plot_fonts.py
"""Select an installed Korean font, or an explicitly supplied font file."""
import os
from pathlib import Path
from matplotlib import font_manager

def korean_font():
    supplied = os.environ.get('SEOUL_FONT_PATH')
    if supplied:
        path = Path(supplied).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'한글 글꼴 파일이 없습니다: {path}')
        font_manager.fontManager.addfont(str(path))
        return font_manager.FontProperties(fname=str(path)).get_name()
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ('Apple SD Gothic Neo', 'Malgun Gothic', 'NanumGothic',
                 'Noto Sans CJK KR', 'Noto Sans KR', 'NanumMyeongjo'):
        if name in installed:
            return name
    raise RuntimeError('한글 글꼴이 필요합니다. 나눔고딕을 설치하거나 SEOUL_FONT_PATH에 글꼴 파일을 지정하세요.')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from plot_fonts import korean_font
plt.rcParams.update({'font.family':korean_font(), 'axes.unicode_minus':False})

gpkg = ROOT/'data/processed/residents/residents.gpkg'
area_boundary = gpd.read_file(gpkg, layer='changsin_sungin_admin').to_crs(5179)
census = gpd.read_file(gpkg, layer='changsin_sungin_elderly').to_crs(5179)
area_boundary = area_boundary[area_boundary.ADM_NM == '창신1동']
census = census[census.ADM_NM == '창신1동'].copy()
centers = census.geometry.representative_point()
assert all(shape.covers(point) for shape, point in zip(census.geometry, centers))
fig, ax = plt.subplots(figsize=(7, 6))
census.plot(ax=ax, column='id', categorical=True, cmap='Pastel1', edgecolor='white', linewidth=1.3)
area_boundary.boundary.plot(ax=ax, color='#314a5b', linewidth=1.8)
centers.plot(ax=ax, color='#203e52', markersize=22)
ax.set_title('창신1동의 집계구와 계산에 사용한 내부 대표점', fontsize=14, pad=14)
ax.axis('off')
fig.text(.5, .035, '색깔 조각 = 집계구 / 점 = 거리 계산 출발점\n자료: SGIS 2025년 2분기 경계. 색은 인구 규모를 뜻하지 않음.',
         ha='center', fontsize=10)
image_path = ROOT/'results/notebook/집계구와_대표점.png'
fig.savefig(image_path, dpi=145, bbox_inches='tight', facecolor='white')
plt.close(fig)
display(Image(filename=str(image_path), width=600))

## 4. 보행망과 이동 상황

OSM에서 보행 가능한 길을 연결합니다. 위치를 가장 가까운 도로 구간에 연결하되
연결 거리가 80m를 넘으면 해당 지점을 연결하지않음,  연결 거리도 이동 비용에 포함

- **왕복:** 집 → 수거지 → 집.
- **정류소 경유 추가거리:** 집 → 수거지 → 정류소 거리에서 집 → 정류소 거리를 뺀 값.

- 정류소는 각 거주지에서 보행거리상 가장 가까운 한 곳으로 고정
- 수거 품목 조건에 따라 `solid`는 전용 수거함+우체통, `liquid`는 전용 수거함을 기존망으로 사용
- 같은 인구를 사용하며 약품별 실제 발생량·배출 행동을 추정한 것은 아님

In [ ]:
%%writefile src/mobility_network.py
"""Reusable walking graph with edge-based connectors and explicit exclusions."""
from collections import Counter, defaultdict
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import networkx as nx
import numpy as np
from pyproj import Transformer
from shapely import LineString, Point, Polygon, STRtree

ROOT = Path(__file__).resolve().parents[1]
TO_XY = Transformer.from_crs(4326, 5179, always_xy=True)
TO_LL = Transformer.from_crs(5179, 4326, always_xy=True)

def xy(record):
    return np.array(TO_XY.transform(record['lon'], record['lat']))

def permitted(tags):
    foot = tags.get('foot', '')
    if foot in ('no', 'private', 'use_sidepath'): return False
    if tags.get('access') in ('no', 'private') and foot not in ('yes', 'designated', 'permissive'): return False
    if tags.get('highway') in ('motorway','motorway_link','trunk','trunk_link','construction','proposed','raceway','busway','platform') and foot not in ('yes','designated'): return False
    return tags.get('area') != 'yes'

class WalkingNetwork:
    def __init__(self, area):
        folder = ROOT / 'data/raw/expanded_osm'
        manifest = json.loads((folder/'manifest.json').read_text())
        self.bbox = manifest['areas'][area]['bbox']
        nodes, ways = {}, {}
        for item in manifest['files']:
            if item['area'] != area: continue
            root = ET.parse(ROOT/item['file']).getroot()
            for n in root.findall('node'):
                key = int(n.attrib['id'])
                if key not in nodes or int(n.attrib.get('version','0')) > nodes[key]['version']:
                    nodes[key] = dict(lon=float(n.attrib['lon']), lat=float(n.attrib['lat']),
                                      version=int(n.attrib.get('version','0')),
                                      tags={t.attrib['k']:t.attrib['v'] for t in n.findall('tag')})
            for way in root.findall('way'):
                key = int(way.attrib['id'])
                if key not in ways or int(way.attrib.get('version','0')) > ways[key]['version']:
                    ways[key] = dict(nodes=[int(n.attrib['ref']) for n in way.findall('nd')],
                                     version=int(way.attrib.get('version','0')),
                                     tags={t.attrib['k']:t.attrib['v'] for t in way.findall('tag')})
        self.positions = {n:xy(r) for n,r in nodes.items()}
        self.graph = nx.DiGraph()
        self.segments = []
        kinds, omitted = Counter(), Counter()
        for wid, way in ways.items():
            tags = way['tags']; kind = tags.get('highway')
            if not kind: continue
            if not permitted(tags): omitted[kind] += 1; continue
            kinds[kind] += 1
            for a,b in zip(way['nodes'][:-1], way['nodes'][1:]):
                if a not in nodes or b not in nodes or a == b: continue
                if any(not permitted(nodes[n]['tags']) for n in (a,b)): continue
                length = float(np.linalg.norm(self.positions[a]-self.positions[b]))
                if length < .001: continue
                direction = tags.get('oneway:foot','no')
                forward, backward = direction != '-1', direction not in ('yes','1','true')
                attr = dict(length=length, steps=kind=='steps', highway=kind, way_id=wid)
                # Coordinate-identical ways do not create crossings unless OSM nodes are shared.
                if forward: self.graph.add_edge(a,b,**attr)
                if backward: self.graph.add_edge(b,a,**attr)
        # One geometry per pair, preserving explicit pedestrian direction restrictions.
        seen = set()
        for a,b,d in list(self.graph.edges(data=True)):
            key = frozenset((a,b))
            if key in seen: continue
            seen.add(key)
            line = LineString([self.positions[a],self.positions[b]])
            self.segments.append((a,b,line,dict(d), self.graph.has_edge(a,b), self.graph.has_edge(b,a)))
        self.lines = [s[2] for s in self.segments]
        self.tree = STRtree(self.lines)
        self.snaps = {}
        w,s,e,n = self.bbox
        # Densified projected extraction boundary for distance-to-edge screening.
        outline = [(w+(e-w)*t,s) for t in np.linspace(0,1,101)] + [(e,s+(n-s)*t) for t in np.linspace(0,1,101)] + [(e-(e-w)*t,n) for t in np.linspace(0,1,101)] + [(w,n-(n-s)*t) for t in np.linspace(0,1,101)]
        self.boundary = Polygon([TO_XY.transform(*p) for p in outline])
        self.audit = dict(area=area,nodes=len(nodes),ways=len(ways),walk_segments=len(self.segments),
                          way_kinds=dict(kinds),omitted_way_kinds=dict(omitted),bbox=self.bbox,
                          explicit_foot_oneway=sum(w['tags'].get('oneway:foot') in ('yes','1','true','-1') for w in ways.values()))

    def attach(self, records, max_offset=80):
        """Call once with all demand points, candidates, bins, and stops."""
        if self.snaps: raise RuntimeError('Attach all records together once, so original segments are split consistently.')
        pending = defaultdict(list)
        for r in records:
            rid = r['id']
            if rid in self.snaps: raise ValueError(f'Duplicate point ID: {rid}')
            p = Point(xy(r)); idx = int(self.tree.nearest(p)); line=self.lines[idx]
            offset = float(line.distance(p)); projection=float(line.project(p))
            self.snaps[rid] = dict(offset_m=offset,matched=offset<=max_offset,segment=idx,
                                   boundary_margin_m=float(p.distance(self.boundary.boundary)))
            if offset > max_offset: continue
            snap = 'SNAP_'+rid; q=line.interpolate(projection)
            self.positions[snap] = np.array([q.x,q.y]); self.positions[rid] = xy(r)
            self.graph.add_edge(rid,snap,length=offset,steps=False,highway='connector',way_id=None)
            self.graph.add_edge(snap,rid,length=offset,steps=False,highway='connector',way_id=None)
            pending[idx].append((projection,snap))
        for idx,entries in pending.items():
            a,b,line,attr,forward,backward = self.segments[idx]
            if self.graph.has_edge(a,b): self.graph.remove_edge(a,b)
            if self.graph.has_edge(b,a): self.graph.remove_edge(b,a)
            chain=sorted([(0.,a),(line.length,b)]+entries,key=lambda p:p[0])
            for (s,n),(t,m) in zip(chain[:-1],chain[1:]):
                edge={**attr,'length':max(0.,t-s)}
                if forward: self.graph.add_edge(n,m,**edge)
                if backward: self.graph.add_edge(m,n,**edge)
        self.audit.update(connected_points=sum(r['matched'] for r in self.snaps.values()),
                          unconnected_points=sum(not r['matched'] for r in self.snaps.values()),
                          graph_nodes=self.graph.number_of_nodes(),graph_arcs=self.graph.number_of_edges())

    def graph_for(self, no_tagged_steps=False):
        if not no_tagged_steps: return self.graph
        return nx.subgraph_view(self.graph, filter_edge=lambda a,b:not self.graph[a][b]['steps'])

    def distances(self, source, targets, reverse=False, no_tagged_steps=False):
        graph=self.graph_for(no_tagged_steps)
        if reverse: graph=graph.reverse(copy=False)
        d = nx.single_source_dijkstra_path_length(graph,source,weight='length') if source in graph else {}
        return np.array([d.get(t,np.inf) for t in targets])

    def nearest(self, sources, targets, reverse=False, no_tagged_steps=False):
        graph=self.graph_for(no_tagged_steps)
        if reverse: graph=graph.reverse(copy=False)
        sources=[s for s in sources if s in graph]
        if not sources: return np.full(len(targets),np.inf), [None]*len(targets)
        d,paths=nx.multi_source_dijkstra(graph,sources,weight='length')
        return np.array([d.get(t,np.inf) for t in targets]),[paths[t][0] if t in paths else None for t in targets]


## 5. 편의점 선정 모형

기존 수거망을 유지하고 최대 K곳을 추가합니다. 각 주민은 기존 시설과 선정 편의점 중
설정한 이동 비용이 가장 작은 곳을 이용한다고 가정

목표: **인구 × 이용 거리의 합 최소화**. 같은 주민의 개선을 중복 집계하지 않음
작은 예제에서는 모든 후보 조합을 직접 열거한 결과와 혼합정수 최적화 해를 비교

In [ ]:
%%writefile src/location_model.py
"""Fixed existing service + optional sites; population-weighted distance minimization.

Costs have the same units within one solve. Each row retains its best existing
facility as column zero. Rows may be resident round trips or hypothetical trips
via a fixed bus stop. Inaccessible candidate assignments are omitted.
"""
import itertools
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix

def solve_location(baseline, candidate_costs, population, k, *, group_weights=None,
                   group_max_mean=None, fixed_sites=(), forbidden_sites=(), time_limit=120):
    baseline=np.asarray(baseline,float); costs=np.asarray(candidate_costs,float)
    weights=np.asarray(population,float)
    n,m=costs.shape
    if baseline.shape!=(n,) or weights.shape!=(n,): raise ValueError('Input dimensions disagree.')
    if np.any(~np.isfinite(baseline)) or np.any(baseline<0): raise ValueError('Baseline must be finite. Report unreachable origins separately.')
    if np.any(~np.isfinite(weights)) or np.any(weights<0) or weights.sum()<=0: raise ValueError('Population weights invalid.')
    if np.any(np.isfinite(costs)&(costs<0)): raise ValueError('Negative travel cost.')
    k=min(int(k),m)
    if k<0: raise ValueError('k must be nonnegative.')
    # Do not assign a demand point to a site worse than its existing alternative.
    rows,cols=np.where(np.isfinite(costs)&(costs<baseline[:,None]-1e-9))
    pairs=list(zip(rows.tolist(),cols.tolist()))
    nv=m+n+len(pairs)
    objective=np.zeros(nv); objective[m:m+n]=weights*baseline/weights.sum()
    for p,(i,j) in enumerate(pairs): objective[m+n+p]=weights[i]*costs[i,j]/weights.sum()
    # Binary site decisions; assignment variables may be continuous because, for
    # a fixed set, assigning to the cheapest site minimizes every cost criterion.
    integrality=np.zeros(nv,int); integrality[:m]=1
    lo=np.zeros(nv); hi=np.ones(nv)
    for j in fixed_sites: lo[j]=1
    for j in forbidden_sites: hi[j]=0
    rr=[]; cc=[]; vv=[]; lower=[]; upper=[]
    def add(terms,lb,ub):
        row=len(lower)
        for col,val in terms:
            rr.append(row);cc.append(col);vv.append(float(val))
        lower.append(lb);upper.append(ub)
    add([(j,1) for j in range(m)],-np.inf,k)
    by_origin=[[(m+i,1)] for i in range(n)]
    for p,(i,j) in enumerate(pairs):
        col=m+n+p
        by_origin[i].append((col,1))
        add([(col,1),(j,-1)],-np.inf,0)
    for terms in by_origin: add(terms,1,1)
    if group_weights is not None:
        gw=np.atleast_2d(np.asarray(group_weights,float)); thresholds=np.atleast_1d(group_max_mean)
        if gw.shape[1]!=n or len(thresholds)!=len(gw): raise ValueError('Fairness dimensions disagree.')
        for g,threshold in zip(gw,thresholds):
            if np.any(~np.isfinite(g)) or np.any(g<0) or g.sum()<=0: raise ValueError('Fairness weights invalid.')
            terms=[(m+i,g[i]*baseline[i]/g.sum()) for i in range(n)]
            terms += [(m+n+p,g[i]*costs[i,j]/g.sum()) for p,(i,j) in enumerate(pairs)]
            add(terms,-np.inf,float(threshold))
    matrix=coo_matrix((vv,(rr,cc)),shape=(len(lower),nv)).tocsc()
    result=milp(objective,integrality=integrality,bounds=Bounds(lo,hi),
                constraints=LinearConstraint(matrix,lower,upper),
                options={'time_limit':time_limit,'mip_rel_gap':1e-7})
    if result.x is None:
        return {'success':False,'status':int(result.status),'message':str(result.message)}
    selected=np.flatnonzero(result.x[:m]>.5)
    after=np.minimum(baseline,costs[:,selected].min(axis=1)) if len(selected) else baseline.copy()
    before=float(np.average(baseline,weights=weights)); mean=float(np.average(after,weights=weights))
    out={'success':True,'proven_optimal':bool(result.success),'status':int(result.status),
         'selected_indices':selected.tolist(),'baseline_mean_m':before,'after_mean_m':mean,
         'reduction_mean_m':before-mean,'reduction_pct':(before-mean)/before*100 if before>0 else 0.,
         'weighted_distance_reduction_person_m':float(np.dot(weights,baseline-after)),
         'mip_gap':float(result.mip_gap),'solver_bound_mean_m':float(result.mip_dual_bound),
         'solver_message':str(result.message),'cost_after':after.tolist()}
    if group_weights is not None:
        out['group_after_mean_m']=[float(np.average(after,weights=g)) for g in gw]
        if any(a>b+1e-4 for a,b in zip(out['group_after_mean_m'],thresholds)):
            raise AssertionError('Selected locations violate fairness bound.')
    if mean>before+1e-6: raise AssertionError('Optional new sites worsened access.')
    if not np.isclose(mean,result.fun,rtol=1e-5,atol=1e-4):
        raise AssertionError('Reported assignment cost does not match selected sites.')
    return out

def validate_model():
    """Small mathematical checks, separate from empirical results."""
    rng=np.random.default_rng(24)
    checked=0
    for _ in range(6):
        baseline=rng.uniform(100,900,7); costs=rng.uniform(0,1000,(7,6)); w=rng.integers(1,100,7)
        costs[0,0]=np.inf
        previous=float(np.average(baseline,weights=w))
        for k in range(4):
            solved=solve_location(baseline,costs,w,k)
            brute=min(float(np.average(np.minimum(baseline,costs[:,s].min(axis=1)),weights=w)) if s else float(np.average(baseline,weights=w))
                      for size in range(k+1) for s in itertools.combinations(range(6),size))
            assert solved['proven_optimal'] and abs(solved['after_mean_m']-brute)<1e-5
            assert solved['after_mean_m']<=previous+1e-6
            previous=solved['after_mean_m']; checked+=1
    return {'enumeration_comparisons':checked,'passed':True,'empirical_results':False}

if __name__=='__main__':
    import json
    print(json.dumps(validate_model()))


In [ ]:
from location_model import validate_model
model_check = validate_model()
assert model_check['passed']
print(f"작은 예제 {model_check['enumeration_comparisons']}개: 완전탐색 결과와 일치")

## 6. 다섯 가지 도입 규모의 기본 분석

0·3·5·10·15곳을 각각 독립적으로 선정, 0곳은 기존망 기준
**2개 지역 × 2개 수거 조건 × 2개 이동 상황 × 5개 도입 규모 = 40개 모형**

In [ ]:
%%writefile src/run_resident_analysis.py
"""Run resident-weighted walking and hypothetical bus-stop detour scenarios.

Requires actual downloaded SGIS inputs from prepare_resident_inputs.py.
There is deliberately no synthetic-data fallback for missing resident files.
"""
import hashlib, json, pickle
from pathlib import Path
import numpy as np
from mobility_network import ROOT, WalkingNetwork
from location_model import solve_location

DATA=ROOT/'data/processed'; RES=DATA/'residents'; OUT=ROOT/'results/resident_analysis'
CONFIGS={'bongcheon':('관악구',['봉천동']),'changsin_sungin':('종로구',['창신동','숭인동'])}

def read(path): return json.loads(path.read_text())
def dump(path,data): path.write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False))
def weighted_quantile(values,weights,q):
    order=np.argsort(values);v=np.asarray(values)[order];w=np.asarray(weights)[order]
    return float(v[np.searchsorted(np.cumsum(w),q*w.sum(),side='left')])

def main():
    for area in CONFIGS:
        if not (RES/f'{area}_population.json').exists():
            raise FileNotFoundError(f'SGIS 주민 인구 원본을 먼저 연결해야 합니다: {area}')
    OUT.mkdir(parents=True,exist_ok=True)
    cvs=read(DATA/'convenience_candidates.json'); bins=read(DATA/'existing_bins.json'); posts=read(DATA/'postboxes.json'); bus=read(DATA/'bus_stops.json')
    summaries=[]
    for area,(gu,dongs) in CONFIGS.items():
        net=WalkingNetwork(area);w,s,e,n=net.bbox
        inside=lambda r:w<=r['lon']<=e and s<=r['lat']<=n
        all_population=read(RES/f'{area}_population.json')
        all_elderly=read(RES/f'{area}_elderly.json') if (RES/f'{area}_elderly.json').exists() else []
        known=[r for r in all_population if r['population'] is not None and r['population']>0]
        elderly_known=[r for r in all_elderly if r['population'] is not None and r.get('population_upper',r['population'])>0]
        pop=[r for r in known if inside(r)];older=[r for r in elderly_known if inside(r)]
        demand=pop+older; N=len(pop)
        candidates=[r for r in cvs if r['district']==gu and r['legal_dong'] in dongs and inside(r) and '슈퍼마켓 가능성' not in r['flags']]
        existing=[r for r in bins+posts if inside(r)]
        sites=existing+candidates;M=len(existing)
        stops=[{**r,'id':'BUS_'+r['id']} for r in bus if inside(r) and r['kind']!='한강선착장']
        net.attach(demand+sites+stops)
        ids=[r['id'] for r in demand];site_ids=[r['id'] for r in sites];stop_ids=[r['id'] for r in stops]
        stop_distance,nearest_stop=net.nearest(stop_ids,ids,reverse=True)
        target_stops=list(dict.fromkeys(b for b in nearest_stop if b is not None))
        stop_lookup={b:i for i,b in enumerate(target_stops)}
        to_sites=np.full((len(demand),len(sites)),np.inf);from_sites=to_sites.copy()
        site_to_stop=np.full((len(target_stops),len(sites)),np.inf)
        undirected=net.audit['explicit_foot_oneway']==0
        network_manifest=read(ROOT/'data/raw/expanded_osm/manifest.json')
        network_sources=[r['sha256'] for r in network_manifest['files'] if r['area']==area]
        signature=hashlib.sha256(json.dumps({'points':demand+sites+stops,'network':net.audit,'sources':network_sources,
          'network_code':hashlib.sha256((ROOT/'src/mobility_network.py').read_bytes()).hexdigest()},ensure_ascii=False,sort_keys=True).encode()).hexdigest()
        cache=OUT/f'{area}_{signature[:16]}_costs.npz'
        if cache.exists():
            arrays=np.load(cache);to_sites=arrays['to_sites'];from_sites=arrays['from_sites'];site_to_stop=arrays['site_to_stop']
        else:
            for j,sid in enumerate(site_ids):
                forward=net.distances(sid,ids+target_stops)
                from_sites[:,j]=forward[:len(ids)];site_to_stop[:,j]=forward[len(ids):]
                to_sites[:,j]=forward[:len(ids)] if undirected else net.distances(sid,ids,reverse=True)
            np.savez_compressed(cache,to_sites=to_sites,from_sites=from_sites,site_to_stop=site_to_stop)
        dump(OUT/f'{area}_cost_context.json',{'demand':demand,'resident_count':N,'candidates':candidates,'existing':existing,
             'stops':stops,'nearest_stop':nearest_stop,'target_stops':target_stops,'cost_cache':cache.name,
             'stop_distance':[float(d) if np.isfinite(d) else None for d in stop_distance]})
        with (OUT/f'{area}_network.pkl').open('wb') as f:pickle.dump(net,f)
        roundtrip=to_sites+from_sites
        chain=np.full_like(roundtrip,np.inf)
        for i,b in enumerate(nearest_stop):
            if b is None: continue
            chain[i]=to_sites[i]+site_to_stop[stop_lookup[b]]-stop_distance[i]
        if np.any(np.isfinite(chain)&(chain<-.01)): raise AssertionError('Detour must be nonnegative by the shortest-path triangle inequality.')
        chain=np.maximum(chain,0)
        scenario_baseline={}
        for medicine in ('solid','liquid'):
            valid=[j for j,r in enumerate(existing) if medicine=='solid' or r['kind']=='전용 수거함']
            if not valid: raise ValueError(f'No existing facilities: {area} {medicine}')
            scenario_baseline[(medicine,'roundtrip')]=np.min(roundtrip[:,valid],axis=1)
            scenario_baseline[(medicine,'stop_detour')]=np.min(chain[:,valid],axis=1)
        pop_weights=np.array([r['population'] if i<N else 0 for i,r in enumerate(demand)],float)
        age_weights=np.array([r['population'] if i>=N else 0 for i,r in enumerate(demand)],float)
        margins=np.array([net.snaps[r['id']]['boundary_margin_m'] for r in demand])
        stop_margins=np.array([net.snaps[b]['boundary_margin_m'] if b else 0 for b in nearest_stop])
        audit={'area':area,'input_population':float(sum(r['population'] for r in known)),
               'population_outside_network':float(sum(r['population'] for r in known if not inside(r))),
               'population_connector_over80m':float(sum(r['population'] for r in pop if not net.snaps[r['id']]['matched'])),
               'elderly_released_population':float(sum(r['population'] for r in elderly_known)),
               'elderly_weight_basis':'집계구 연령·총인구에서 산출한 하한',
               'candidate_count':len(candidates),'network':net.audit,'cohorts':{}}
        for trip_type,costs in [('roundtrip',roundtrip),('stop_detour',chain)]:
            finite=np.isfinite(scenario_baseline[('solid',trip_type)])&np.isfinite(scenario_baseline[('liquid',trip_type)])
            if trip_type=='roundtrip':
                # Any route that exits and returns must travel to the extraction edge twice.
                safe=finite & (scenario_baseline[('liquid',trip_type)]<2*margins)
            else:
                # A path leaving the extract must travel origin→edge and edge→fixed stop.
                full_path=scenario_baseline[('liquid',trip_type)]+stop_distance
                safe=finite & (full_path<margins+stop_margins) & (stop_distance<margins)
            included=np.flatnonzero(safe)
            weights=pop_weights[included];ages=age_weights[included]
            if weights.sum()<=0: raise ValueError('No resident population in the verified comparison cohort.')
            audit['cohorts'][trip_type]={'population':float(weights.sum()),'elderly_released':float(ages.sum()),
                                      'resident_points':int(sum(i<N for i in included)),
                                      'population_share_of_input_pct':float(weights.sum()/audit['input_population']*100)}
            c=costs[np.ix_(included,range(M,len(sites)))]
            for medicine in ('solid','liquid'):
                base=scenario_baseline[(medicine,trip_type)][included]
                for k in (0,3,5,10,15):
                    solved=solve_location(base,c,weights,k,time_limit=90)
                    if not solved['success']: raise RuntimeError(solved)
                    after=np.array(solved.pop('cost_after'))
                    solved.update(area=area,medicine=medicine,trip_type=trip_type,k=k,
                                  population=float(weights.sum()),
                                  baseline_p90_m=weighted_quantile(base,weights,.9),after_p90_m=weighted_quantile(after,weights,.9),
                                  selected_sites=[candidates[j] for j in solved['selected_indices']],
                                  elderly_mean_before_m=float(np.average(base,weights=ages)) if ages.sum()>0 else None,
                                  elderly_mean_after_m=float(np.average(after,weights=ages)) if ages.sum()>0 else None,
                                  trip_assumption='각 주민 1회 왕복 배출의 잠재 보행거리' if trip_type=='roundtrip' else '각 거주지에서 보행거리상 가장 가까운 정류소로 가는 가정 경로의 추가거리')
                    # Leave-one-out is a participation failure scenario, not a measured probability.
                    losses=[]
                    for missing in solved['selected_indices']:
                        remaining=[j for j in solved['selected_indices'] if j!=missing]
                        fallback=np.minimum(base,c[:,remaining].min(axis=1)) if remaining else base.copy()
                        losses.append({'site_id':candidates[missing]['id'],'mean_added_distance_m':float(np.average(fallback-after,weights=weights))})
                    solved['single_store_dropout']=losses
                    summaries.append(solved)
                    detail=[{**demand[i],'weight_in_total_objective':float(weights[p]),'baseline_m':float(base[p]),'after_m':float(after[p])} for p,i in enumerate(included)]
                    dump(OUT/f'{area}_{medicine}_{trip_type}_k{k}_origins.json',detail)
            # Save the exact common cohort used for both medicine categories.
        dump(OUT/f'{area}_audit.json',audit)
        print(area,audit['cohorts'],flush=True)
    dump(OUT/'summary.json',summaries)

if __name__=='__main__': main()


In [ ]:
if REBUILD_ROUTES:
    for cached in (ROOT/'results/resident_analysis').glob('*_costs.npz'):
        cached.unlink()
    print('거리 행렬 캐시를 지우고 경로부터 다시 계산합니다.')
from run_resident_analysis import main as run_basic_analysis
run_stage('04_기본_40개_모형', run_basic_analysis)
summary = read_json('results/resident_analysis/summary.json')
assert len(summary) == 40
assert all(row['proven_optimal'] for row in summary)
cohort_table = []
for area in area_names:
    audit = read_json(f'results/resident_analysis/{area}_audit.json')
    cohort_table.append({'지역':area_names[area], '편의점_후보수':audit['candidate_count'],
                         '거리비교_인구':audit['cohorts']['roundtrip']['population'],
                         '출발_격자수':audit['cohorts']['roundtrip']['resident_points']})
display(pd.DataFrame(cohort_table))

In [ ]:
rows = [{'지역':area_names[r['area']], '수거조건':'물약류' if r['medicine']=='liquid' else '알약류',
         '추가_편의점수':r['k'], '평균_왕복거리_m':r['after_mean_m'], '기존대비_감소율_pct':r['reduction_pct']}
        for r in summary if r['trip_type']=='roundtrip']
scale_table = pd.DataFrame(rows)
display(scale_table.pivot(index='추가_편의점수', columns=['지역','수거조건'], values='평균_왕복거리_m').round(1))
print('각 열 안에서 도입 수에 따른 거리 변화를 비교합니다. 규모마다 선정 점포 조합은 달라질 수 있습니다.')

## 7. 고령층 우선안과 추가 검토

- 고령층 평균거리는 **Σ(집계구별 65세 이상 인구 수 × 대표점의 이용 거리) / 고령인구 총합**입니다.

- 전체 주민의 개선 효과를 효율안의 90%·95%·100% 이상 유지하면서 이 평균거리를 최소화
보고서의 대표 조건은 95%



In [ ]:
%%writefile src/run_resident_extensions.py
"""Equity, drop-out replacement, age uncertainty, and tagged-stair sensitivity."""
import json,pickle
import numpy as np
from mobility_network import ROOT
from location_model import solve_location

OUT=ROOT/'results/resident_analysis'
def read(p):return json.loads(p.read_text())
def dump(p,x):p.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False))
def avg(v,w):return float(np.average(v,weights=w))
def after_cost(base,c,selected):return np.minimum(base,c[:,selected].min(axis=1)) if selected else base.copy()

def main():
    summary=read(OUT/'summary.json');ext=[]
    for area in ['bongcheon','changsin_sungin']:
        context=read(OUT/f'{area}_cost_context.json');rows=context['demand'];N=context['resident_count'];M=len(context['existing'])
        data=np.load(OUT/context['cost_cache']);roundtrip=data['to_sites']+data['from_sites']
        stop_index={b:i for i,b in enumerate(context['target_stops'])}
        chain=np.full_like(roundtrip,np.inf)
        for i,b in enumerate(context['nearest_stop']):
            if b is not None:chain[i]=data['to_sites'][i]+data['site_to_stop'][stop_index[b]]-context['stop_distance'][i]
        chain=np.maximum(chain,0)
        all_ids={r['id']:i for i,r in enumerate(rows)}
        for trip,costs in [('roundtrip',roundtrip),('stop_detour',chain)]:
            for medicine in ('solid','liquid'):
                efficient=next(r for r in summary if r['area']==area and r['trip_type']==trip and r['medicine']==medicine and r['k']==5)
                cohort=read(OUT/f'{area}_{medicine}_{trip}_k5_origins.json')
                idx=np.array([all_ids[r['id']] for r in cohort]);base=np.array([r['baseline_m'] for r in cohort]);c=costs[idx,M:]
                pop=np.array([r['population'] if r['source_kind']=='grid100m' else 0 for r in cohort])
                low=np.array([r['population_lower'] if r['source_kind']=='census' else 0 for r in cohort])
                high=np.array([r['population_upper'] if r['source_kind']=='census' else 0 for r in cohort])
                E=efficient['selected_indices'];eff_after=after_cost(base,c,E)
                B=avg(base,pop);Emean=avg(eff_after,pop)
                fair=[]
                for retained in (.90,.95,1.):
                    limit=B-retained*(B-Emean)+1e-6
                    solved=solve_location(base,c,low,5,group_weights=pop,group_max_mean=[limit],time_limit=90)
                    if not solved['success']:raise RuntimeError(solved)
                    F=solved['selected_indices'];v=after_cost(base,c,F)
                    fair.append({'retained_total_benefit_fraction':retained,'selected_indices':F,
                                 'population_mean_m':avg(v,pop),'elderly_lower_weighted_mean_m':avg(v,low),
                                 'elderly_upper_weighted_mean_m':avg(v,high),'proven_optimal':solved['proven_optimal'],
                                 'mip_gap':solved['mip_gap'],'overlap_with_efficiency':len(set(E)&set(F))})
                upper_solve=solve_location(base,c,high,5,group_weights=pop,
                    group_max_mean=[B-.95*(B-Emean)+1e-6],time_limit=90)
                if not upper_solve['success']:raise RuntimeError(upper_solve)
                replacements=[]
                for missing in E:
                    fixed=[j for j in E if j!=missing];lost=after_cost(base,c,fixed)
                    eligible=[j for j in range(c.shape[1]) if j not in E]
                    j=min(eligible,key=lambda j:avg(np.minimum(lost,c[:,j]),pop))
                    replacements.append({'dropped_index':missing,'replacement_index':j,
                                         'dropout_mean_m':avg(lost,pop),'replacement_mean_m':avg(np.minimum(lost,c[:,j]),pop)})
                largest=int(np.argmax(pop));wo=pop.copy();wo[largest]=0
                outlier=solve_location(base,c,wo,5,time_limit=90)
                restricted=[j for j,r in enumerate(context['candidates']) if any(f in r['flags'] for f in ['시설 내부 출입 확인','층 정보 확인'])]
                restricted_result=solve_location(base,c,pop,5,forbidden_sites=restricted,time_limit=90)
                ext.append({'area':area,'trip_type':trip,'medicine':medicine,
                            'population_before_m':B,'population_efficient_m':Emean,
                            'elderly_before_lower_m':avg(base,low),'elderly_efficient_lower_m':avg(eff_after,low),
                            'elderly_before_upper_m':avg(base,high),'elderly_efficient_upper_m':avg(eff_after,high),
                            'fairness':fair,'elderly_upper_95_selected':upper_solve['selected_indices'],
                            'age_weight_interval_note':'하한·상한 가중치 두 시나리오이며 가능한 모든 인구 배치의 최악값을 뜻하지 않음.',
                            'dropout_replacements':replacements,
                            'largest_grid_sensitivity':{'grid_id':cohort[largest]['id'],'population':float(pop[largest]),
                                'share_pct':float(pop[largest]/pop.sum()*100),'selected':outlier['selected_indices'],
                                'overlap_with_main':len(set(E)&set(outlier['selected_indices'])),
                                'after_mean_on_omitted_sample_m':outlier['after_mean_m']},
                            'restricted_candidates':{'excluded_count':len(restricted),'selected':restricted_result['selected_indices'],
                                'after_mean_m':restricted_result['after_mean_m'],'reduction_pct':restricted_result['reduction_pct']}})
        # Evaluate the already-selected five stores when all OSM-tagged stairs are removed.
        with (OUT/f'{area}_network.pkl').open('rb') as f:net=pickle.load(f)
        selected_union=sorted(set(j for r in summary if r['area']==area and r['trip_type']=='roundtrip' and r['k']==5 for j in r['selected_indices']))
        all_sites=context['existing']+context['candidates']
        cols=list(range(M))+[M+j for j in selected_union]
        no_steps=np.full((len(rows),len(cols)),np.inf)
        for q,j in enumerate(cols):
            sid=all_sites[j]['id'];fwd=net.distances(sid,[r['id'] for r in rows],no_tagged_steps=True)
            rev=fwd if net.audit['explicit_foot_oneway']==0 else net.distances(sid,[r['id'] for r in rows],reverse=True,no_tagged_steps=True)
            no_steps[:,q]=fwd+rev
        stairs=[]
        for medicine in ('solid','liquid'):
            entry=next(r for r in summary if r['area']==area and r['trip_type']=='roundtrip' and r['medicine']==medicine and r['k']==5)
            cohort=read(OUT/f'{area}_{medicine}_roundtrip_k5_origins.json');idx=np.array([all_ids[r['id']] for r in cohort])
            pop=np.array([r['population'] if r['source_kind']=='grid100m' else 0 for r in cohort]);base=np.array([r['baseline_m'] for r in cohort]);old_after=np.array([r['after_m'] for r in cohort])
            existing_cols=[j for j,r in enumerate(context['existing']) if medicine=='solid' or r['kind']=='전용 수거함']
            base_new=no_steps[np.ix_(idx,existing_cols)].min(axis=1)
            new_cols=existing_cols+[cols.index(M+j) for j in entry['selected_indices']]
            after_new=no_steps[np.ix_(idx,new_cols)].min(axis=1)
            reachable=np.isfinite(base_new)&np.isfinite(after_new)
            margins=np.array([net.snaps[r['id']]['boundary_margin_m'] for r in cohort])
            finite=reachable & (base_new<2*margins)
            if np.any(base_new[finite]+.01<base[finite]) or np.any(after_new[finite]+.01<old_after[finite]):raise AssertionError('Removing steps shortened a route.')
            stairs.append({'medicine':medicine,'common_reachable_population':float(pop[finite].sum()),
                           'boundary_excluded_population':float(pop[reachable&~finite].sum()),
                           'baseline_unreachable_population':float(pop[~np.isfinite(base_new)].sum()),
                           'after_unreachable_population':float(pop[~np.isfinite(after_new)].sum()),
                           'normal_before_common_m':avg(base[finite],pop[finite]),'normal_after_common_m':avg(old_after[finite],pop[finite]),
                           'no_tagged_steps_before_common_m':avg(base_new[finite],pop[finite]),'no_tagged_steps_after_common_m':avg(after_new[finite],pop[finite]),
                           'same_selected_stores':True,'reoptimized':False})
        dump(OUT/f'{area}_stair_sensitivity.json',stairs)
        print(area,'extensions complete',flush=True)
    dump(OUT/'extensions.json',ext)

if __name__=='__main__':main()


In [ ]:
from run_resident_extensions import main as run_extensions
run_stage('05_고령층과_민감도', run_extensions)
extensions = read_json('results/resident_analysis/extensions.json')
age_table = []
for r in extensions:
    if r['medicine'] != 'liquid' or r['trip_type'] != 'roundtrip':
        continue
    fair = next(f for f in r['fairness'] if f['retained_total_benefit_fraction'] == .95)
    age_table.append({'지역':area_names[r['area']], '효율안_전체평균_m':r['population_efficient_m'],
                      '고령층우선안_전체평균_m':fair['population_mean_m'],
                      '효율안_고령평균_m':r['elderly_efficient_lower_m'],
                      '고령층우선안_고령평균_m':fair['elderly_lower_weighted_mean_m'],
                      '같은_선정점포수':fair['overlap_with_efficiency']})
display(pd.DataFrame(age_table).round(1))
print('물약류·왕복·5곳·전체 개선 95% 유지. 전체 평균은 100m 격자, 고령 평균은 집계구 대표점 기준')

## 8. 지도와 결과 그림 생성

아래 코드는 결과를 출발점 자료에서 다시 집계해 검증하고, 지도와 그래프 및 HTML 검토 자료를 만듬
생성 후 노트북에서 도입 규모 비교, 5곳 선정 지도, 정류소 경유, 고령층 비교를 차례로 확인

In [ ]:
%%writefile src/build_resident_report.py
"""Validate completed resident models and publish a local, self-contained briefing."""
import html, json, pickle, hashlib
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, colors
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mobility_network import ROOT, xy

R=ROOT/'results/resident_analysis'; F=R/'figures'; F.mkdir(parents=True,exist_ok=True)
def read(p): return json.loads(p.read_text())
def dump(p,x): p.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False))
def mean(v,w): return float(np.average(v,weights=w))
def esc(x): return html.escape(str(x))
from plot_fonts import korean_font
plt.rcParams.update({'font.family':korean_font(),'axes.unicode_minus':False,'font.size':12,
                    'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
LABELS={'bongcheon':'봉천동 일대','changsin_sungin':'창신·숭인동'}
S=read(R/'summary.json'); E=read(R/'extensions.json'); INPUT=read(ROOT/'results/resident_input_audit.json')
C={a:read(R/f'{a}_cost_context.json') for a in LABELS}
A={a:read(R/f'{a}_audit.json') for a in LABELS}
def result(a,med='liquid',trip='roundtrip',k=5):
    return next(s for s in S if (s['area'],s['medicine'],s['trip_type'],s['k'])==(a,med,trip,k))
def extension(a,med='liquid',trip='roundtrip'):
    return next(s for s in E if (s['area'],s['medicine'],s['trip_type'])==(a,med,trip))
def save(fig,name):
    fig.savefig(F/f'{name}.png',dpi=190,bbox_inches='tight',facecolor='white')
    fig.savefig(F/f'{name}.svg',bbox_inches='tight',facecolor='white')
    plt.close(fig)

# Results are independently recomputed from origin records; no reported number is hand-entered.
checks=[]; dongs=[]; cross=[]; coverage=[]; network_scope=[]
all_candidates=read(ROOT/'data/processed/convenience_candidates.json')
for area in LABELS:
    admin=gpd.read_file(ROOT/'data/processed/residents/residents.gpkg',layer=f'{area}_admin')
    bound=admin.to_crs(4326).geometry.union_all()
    ids={r['id'] for r in all_candidates if bound.covers(Point(r['lon'],r['lat'])) and '슈퍼마켓 가능성' not in r['flags']}
    assert ids=={r['id'] for r in C[area]['candidates']}, 'Demand/candidate boundary mismatch'
    network_scope.append({'area':area,'candidate_boundary_matches_demand_admin_union':True,'candidate_count':len(ids)})
    for med in ('solid','liquid'):
        for trip in ('roundtrip','stop_detour'):
            prev=np.inf
            for k in (0,3,5,10,15):
                s=result(area,med,trip,k);rows=read(R/f'{area}_{med}_{trip}_k{k}_origins.json')
                p=[r for r in rows if r['source_kind']=='grid100m'];w=np.array([r['population'] for r in p]);b=np.array([r['baseline_m'] for r in p]);v=np.array([r['after_m'] for r in p])
                assert s['proven_optimal'] and s['mip_gap']<=1.1e-7
                assert np.all(v>=0) and np.all(v<=b+1e-6)
                assert np.isclose(mean(v,w),s['after_mean_m']) and np.isclose(mean(b,w),s['baseline_mean_m'])
                assert s['after_mean_m']<=prev+1e-6;prev=s['after_mean_m']
                if k==5 and trip=='roundtrip':
                    for radius in (200,350,500):
                        coverage.append({'area':area,'medicine':med,'walking_oneway_threshold_m':radius,
                            'before_pct':float(w[b<=2*radius].sum()/w.sum()*100),'after_pct':float(w[v<=2*radius].sum()/w.sum()*100)})
                    if med=='liquid':
                        for dong in sorted({r['dong'] for r in p}):
                            z=[r for r in p if r['dong']==dong];ww=[r['population'] for r in z]
                            dongs.append({'area':area,'dong':dong,'population':sum(ww),
                                          'before_m':mean([r['baseline_m'] for r in z],ww),'after_m':mean([r['after_m'] for r in z],ww)})
                checks.append({'area':area,'medicine':med,'trip':trip,'k':k,'passed':True})
    context=C[area];data=np.load(R/context['cost_cache']);M=len(context['existing'])
    stop_idx={b:i for i,b in enumerate(context['target_stops'])};idx_by_id={r['id']:i for i,r in enumerate(context['demand'])}
    for med in ('solid','liquid'):
        cohort=read(R/f'{area}_{med}_stop_detour_k5_origins.json')
        pop=[r for r in cohort if r['source_kind']=='grid100m'];idx=[idx_by_id[r['id']] for r in pop]
        c=np.array([data['to_sites'][i,M:]+data['site_to_stop'][stop_idx[context['nearest_stop'][i]],M:]-context['stop_distance'][i] for i in idx])
        c=np.maximum(c,0);base=np.array([r['baseline_m'] for r in pop]);w=np.array([r['population'] for r in pop])
        round_sel=result(area,med)['selected_indices'];chain_sel=result(area,med,'stop_detour')['selected_indices']
        same=mean(np.minimum(base,c[:,round_sel].min(axis=1)),w)
        cross.append({'area':area,'medicine':med,'before_detour_m':mean(base,w),'roundtrip_selected_detour_m':same,
                      'detour_selected_detour_m':result(area,med,'stop_detour')['after_mean_m'],
                      'common_selected_stores':len(set(round_sel)&set(chain_sel))})
dump(R/'validation.json',{'models_checked':len(checks),'all_passed':True,'scope':network_scope,'checks':checks})
dump(R/'interpretation_metrics.json',{'districts':dongs,'same_store_cross_evaluation':cross,'walking_coverage':coverage})

# Maps: same scale and same resident cohort in the before/after panels.
for area,label in LABELS.items():
    admin=gpd.read_file(ROOT/'data/processed/residents/residents.gpkg',layer=f'{area}_admin')
    grid=gpd.read_file(ROOT/'data/processed/residents/residents.gpkg',layer=f'{area}_grid100m')
    rows=read(R/f'{area}_liquid_roundtrip_k5_origins.json');lookup={r['source_id']:r for r in rows if r['source_kind']=='grid100m'}
    grid['before']=grid['id'].map(lambda s:lookup[s]['baseline_m'] if s in lookup else np.nan)
    grid['after']=grid['id'].map(lambda s:lookup[s]['after_m'] if s in lookup else np.nan)
    with (R/f'{area}_network.pkl').open('rb') as f: net=pickle.load(f)
    roads=[np.asarray(s[2].coords) for s in net.segments]
    positive=grid[grid.population>0];bounds=positive.total_bounds;pad=130
    norm=colors.Normalize(0,2000,clip=True);cmap=plt.get_cmap('YlOrRd')
    fig,axes=plt.subplots(1,2,figsize=(13.5,7.0))
    fig.subplots_adjust(left=.01,right=.9,bottom=.16,top=.93,wspace=.05)
    selected=result(area)['selected_sites']
    for ax,col,title in zip(axes,['before','after'],['기존 수거망','편의점 5곳 추가']):
        admin.plot(ax=ax,color='#f7f7f5',edgecolor='#b6c1c6',linewidth=.6,zorder=0)
        ax.add_collection(LineCollection(roads,colors='#ced7dc',linewidths=.27,zorder=1))
        grid[grid.population.isna()].plot(ax=ax,color='#e6e8ea',edgecolor='none',zorder=2)
        grid[grid[col].notna()].plot(ax=ax,column=col,cmap=cmap,norm=norm,edgecolor='white',linewidth=.18,zorder=3)
        excluded=grid[(grid.population>0)&grid[col].isna()]
        if len(excluded): excluded.plot(ax=ax,facecolor='none',edgecolor='#626f78',hatch='///',linewidth=.8,zorder=4)
        bins=[r for r in C[area]['existing'] if r['kind']=='전용 수거함'];X=np.array([xy(r) for r in bins])
        ax.scatter(X[:,0],X[:,1],s=26,marker='s',c='#174b68',edgecolors='white',linewidth=.65,zorder=6)
        if col=='after':
            for n,r in enumerate(selected,1):
                x,y=xy(r);ax.scatter(x,y,s=250,c='#ffffff',edgecolors='#006a64',linewidths=2,zorder=8)
                ax.text(x,y,str(n),ha='center',va='center',color='#006a64',fontsize=11,fontweight='bold',zorder=9)
        for _,r in admin.iterrows():
            p=r.geometry.representative_point()
            if bounds[0]<p.x<bounds[2] and bounds[1]<p.y<bounds[3]:
                ax.text(p.x,p.y,r.ADM_NM,fontsize=8,color='#233e4a',ha='center',zorder=5,bbox=dict(facecolor='white',edgecolor='none',alpha=.65,pad=.5))
        ax.set_xlim(bounds[0]-pad,bounds[2]+pad);ax.set_ylim(bounds[1]-pad,bounds[3]+pad);ax.set_aspect('equal');ax.axis('off');ax.set_title(title,fontsize=16,pad=12)
        length=500 if area=='bongcheon' else 250
        sx=bounds[0]+10;sy=bounds[1]-70;ax.plot([sx,sx+length],[sy,sy],color='#294652',lw=2);ax.text(sx+length/2,sy+30,f'{length} m',ha='center',fontsize=9)
    color_axis=fig.add_axes([.92,.23,.016,.58])
    cb=fig.colorbar(plt.cm.ScalarMappable(norm=norm,cmap=cmap),cax=color_axis,extend='max')
    cb.set_label('물약류 배출 왕복 보행거리 (m)',labelpad=10)
    fig.legend(handles=[Line2D([0],[0],ls='',marker='s',color='#174b68',label='기존 전용 수거함'),
                        Line2D([0],[0],ls='',marker='o',mfc='white',mec='#006a64',label='선정 편의점 (번호는 표와 연결)'),
                        Patch(facecolor='#e6e8ea',label='인구 미공개 격자'),
                        Patch(facecolor='white',edgecolor='#626f78',hatch='///',label='경로 연결이 없어 비교에서 제외')],loc='lower center',bbox_to_anchor=(.5,.055),ncol=2,fontsize=10,frameon=False)
    fig.text(.5,.022,'자료: SGIS(2024 인구·2025 경계), 스마트서울맵, 소상공인시장진흥공단, © OpenStreetMap contributors / ODbL',ha='center',fontsize=9,color='#61727b')
    save(fig,f'01_{area}_왕복거리_전후지도')

# Number of sites, travel situations, and elderly tradeoff.
fig,axes=plt.subplots(1,2,figsize=(12,4.8),layout='constrained')
for ax,(a,l) in zip(axes,LABELS.items()):
    for med,c,t in [('liquid','#c05130','물약류'),('solid','#218681','알약류')]:
        rr=[result(a,med,k=k) for k in (0,3,5,10,15)];ax.plot([r['k'] for r in rr],[r['after_mean_m'] for r in rr],'-o',color=c,label=t,lw=2.5)
        for r in rr: ax.annotate(f"{r['after_mean_m']:.0f}",(r['k'],r['after_mean_m']),xytext=(0,9 if med=='liquid' else -18),textcoords='offset points',ha='center',fontsize=10,color=c)
    ax.set_title(l);ax.set_xlabel('추가 편의점 수 (곳)');ax.set_ylabel('인구 가중 평균 왕복거리 (m)');ax.set_xticks([0,3,5,10,15]);ax.grid(axis='y',alpha=.18);ax.legend(frameon=False);ax.set_ylim(0,1250)
save(fig,'02_도입규모와_왕복거리')

fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
for ax,(a,l) in zip(axes,LABELS.items()):
    r=next(r for r in cross if r['area']==a and r['medicine']=='liquid')
    vals=[r['before_detour_m'],r['roundtrip_selected_detour_m'],r['detour_selected_detour_m']]
    bars=ax.bar(['기존망','왕복거리로 선정한\n같은 5곳','경유거리로\n다시 선정한 5곳'],vals,color=['#97a8b4','#268781','#316bac'],width=.62)
    ax.bar_label(bars,fmt='%.0f m',padding=5);ax.set_title(l);ax.set_ylabel('정류소 경유 추가거리 (m)');ax.set_ylim(0,1000);ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
save(fig,'03_같은점포의_정류소경유효과')

fig,axes=plt.subplots(1,2,figsize=(12,4.8),layout='constrained')
for ax,(a,l) in zip(axes,LABELS.items()):
    e=extension(a);rr=e['fairness'];xx=[e['population_efficient_m']]+[r['population_mean_m'] for r in rr[:2]];yy=[e['elderly_efficient_lower_m']]+[r['elderly_lower_weighted_mean_m'] for r in rr[:2]]
    ax.scatter(xx,yy,s=[120,85,110],color=['#266882','#b8b8b8','#148579'],zorder=3)
    for x,y,t,offset in zip(xx,yy,['전체 거리 우선','효과 90% 유지','효과 95% 유지'],[(6,7),(8,-32),(6,9)]):
        ax.annotate(f'{t}\n({x:.0f}, {y:.0f})',(x,y),xytext=offset,textcoords='offset points',fontsize=10)
    ax.set_title(l);ax.set_xlabel('전체 인구 평균 왕복거리 (m)');ax.set_ylabel('65세 이상 평균 왕복거리 (m)');ax.grid(alpha=.2)
    ax.set_xlim(min(xx)-10,max(xx)+43);ax.set_ylim(min(yy)-13,max(yy)+20)
fig.text(.5,-.035,'물약류 · 5곳 추가 · 전체 인구는 100m 격자, 고령인구는 집계구의 연령별 인구 하한 가중치. 자료: 본 연구 계산.',ha='center',fontsize=9,color='#61727b')
save(fig,'04_전체효과와_고령층개선')

fig,axes=plt.subplots(1,2,figsize=(12,5.5),layout='constrained')
for ax,(a,l) in zip(axes,LABELS.items()):
    rr=sorted([r for r in dongs if r['area']==a],key=lambda r:r['before_m']-r['after_m'])
    for y,r in enumerate(rr):
        ax.plot([r['after_m'],r['before_m']],[y,y],color='#becdd4',lw=4)
        ax.scatter(r['before_m'],y,color='#98a9b3',s=55,zorder=3);ax.scatter(r['after_m'],y,color='#218681',s=55,zorder=3)
    ax.set_yticks(range(len(rr)),[r['dong'] for r in rr]);ax.set_title(l);ax.set_xlabel('동별 인구 가중 평균 왕복거리 (m)');ax.grid(axis='x',alpha=.15);ax.set_xlim(0,1900)
fig.legend(handles=[Line2D([0],[0],ls='',marker='o',color='#98a9b3',label='기존망'),Line2D([0],[0],ls='',marker='o',color='#218681',label='5곳 추가')],loc='outside lower center',ncol=2,frameon=False)
save(fig,'05_동별_개선효과')

def img(name,alt): return f'<figure><a href="figures/{name}.png" target="_blank"><img src="figures/{name}.png" alt="{esc(alt)}"></a><figcaption><a href="figures/{name}.png" download>PNG 저장</a> · <a href="figures/{name}.svg" download>SVG 저장</a></figcaption></figure>'
def table(headers,rows):return '<div class="scroll"><table><thead><tr>'+''.join(f'<th>{esc(h)}</th>' for h in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join(f'<td>{v}</td>' for v in row)+'</tr>' for row in rows)+'</tbody></table></div>'
headline_rows=[];cards=[]
for a,l in LABELS.items():
    s=result(a);cards.append(f'<div class="card"><span>{l} · 물약류 왕복</span><strong>{s["baseline_mean_m"]:.0f} → {s["after_mean_m"]:.0f} m</strong><em>5곳 추가로 {s["reduction_pct"]:.1f}% 감소</em></div>')
    for med,t in [('solid','알약류'),('liquid','물약류')]:
        s=result(a,med);headline_rows.append([l,t,f'{s["baseline_mean_m"]:.0f} m',f'{s["after_mean_m"]:.0f} m',f'{s["reduction_pct"]:.1f}%',f'{s["baseline_p90_m"]:.0f} → {s["after_p90_m"]:.0f} m'])
maps=''.join(f'<h3>{l}</h3>'+img(f'01_{a}_왕복거리_전후지도',f'{l} 물약류 왕복거리 전후 비교') for a,l in LABELS.items())
sites=''
for a,l in LABELS.items():
    s=result(a);sites+=f'<h3>{l}</h3>'+table(['지도 번호','선정 후보','행정동','주소','자료 확인 항목'],[[str(i),esc(r['name']),esc(r['dong']),esc(r['address']),esc(' · '.join(r['flags']) or '별도 표시 없음')] for i,r in enumerate(s['selected_sites'],1)])
fair_rows=[];fair_notes=[];risk_rows=[]
for a,l in LABELS.items():
    e=extension(a);f=e['fairness'][1];delta=e['elderly_efficient_lower_m']-f['elderly_lower_weighted_mean_m']
    fair_rows.append([l,f'{e["population_efficient_m"]:.0f} → {f["population_mean_m"]:.0f} m',f'{e["elderly_efficient_lower_m"]:.0f} → {f["elderly_lower_weighted_mean_m"]:.0f} m',f'{5-f["overlap_with_efficiency"]}곳 교체'])
    overlap=len(set(f['selected_indices'])&set(e['elderly_upper_95_selected']))
    fair_notes.append(f'{l}은 고령인구 하한·상한 가중치를 바꿨을 때 5곳 중 {overlap}곳이 같았습니다.')
    worst=max(e['dropout_replacements'],key=lambda r:r['dropout_mean_m']);cs=C[a]['candidates']
    risk_rows.append([l,esc(cs[worst['dropped_index']]['name']),f'{e["population_efficient_m"]:.0f} → {worst["dropout_mean_m"]:.0f} m',esc(cs[worst['replacement_index']]['name']),f'{worst["replacement_mean_m"]:.0f} m'])
stairs_rows=[]
for a,l in LABELS.items():
    r=next(x for x in read(R/f'{a}_stair_sensitivity.json') if x['medicine']=='liquid')
    stairs_rows.append([l,f'{r["common_reachable_population"]:,.0f}',f'{r["normal_before_common_m"]:.0f} → {r["normal_after_common_m"]:.0f} m',f'{r["no_tagged_steps_before_common_m"]:.0f} → {r["no_tagged_steps_after_common_m"]:.0f} m',f'{r["baseline_unreachable_population"]:,.0f}'])

sources=[('SGIS 자료제공','https://sgis.mods.go.kr/view/pss/openDataIntrcn','2024 인구·연령 통계, 2025년 2분기 행정동·집계구 경계, 100m 격자. 승인 자료 4개 ZIP 확보.'),
('소상공인시장진흥공단 상가정보','https://www.data.go.kr/data/15083033/fileData.do','2026.6.30 편의점 분류 자료. 동일 좌표 정리 및 슈퍼마켓 가능 후보 제외.'),
('스마트서울맵','https://map.seoul.go.kr/smgis2/','2026.10.4 조회, 폐의약품 전용수거함 테마.'),
('인터넷우체국 우체통 찾기','https://m.epost.go.kr/mobile/postFind/postMapSearch.jsp','2026.10.4 조회. 알약류 배출 시나리오에 사용.'),
('서울시 버스정류소 위치정보','https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do','2026.9.2 자료. 한강선착장 제외.'),
('© OpenStreetMap contributors / ODbL','https://www.openstreetmap.org/copyright','2026.10.4 보행망. 전체 33개 분할 추출 파일을 중복 정리해 연결.'),
('서울시 폐의약품 배출 안내','https://mediahub.seoul.go.kr/archives/2015766','2025.10.24. 우체통의 물약류 수거 제외를 반영.'),
('Hakimi (1964), Optimum Locations of Switching Centers','https://doi.org/10.1287/opre.12.3.450','도로망 위에서 수요 가중 거리를 최소화하는 입지 분석의 방법론적 배경.')]
source_html='<ol>'+''.join(f'<li><a href="{u}">{t}</a> — {d}</li>' for t,u,d in sources)+'</ol>'
page='''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>서울 폐의약품 배출 이동부담 — 주민 인구를 반영한 최적화</title>
<style>body{margin:0;background:#f2f5f6;color:#223b48;font:17px/1.75 -apple-system,BlinkMacSystemFont,'Apple SD Gothic Neo',sans-serif}main{max-width:1120px;margin:auto;padding:40px 22px 70px}header,section{background:#fff;padding:30px 36px;margin-bottom:20px;border-radius:12px}h1{font-size:34px;line-height:1.35;letter-spacing:-.8px}h2{font-size:25px;margin-top:0}h3{font-size:20px;margin-top:28px}.kicker{color:#177f79;font-size:14px;letter-spacing:.4px}.lead{font-size:21px}.cards{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:24px 0}.card{background:#eef7f5;border-radius:10px;padding:22px}.card span,.card strong,.card em{display:block}.card strong{font-size:30px;color:#125e58}.card em{font-style:normal;color:#237e74}.note,figcaption{font-size:14px;color:#5b707a}a{color:#186e95}nav{display:flex;gap:18px;flex-wrap:wrap;padding:12px 0}figure{margin:24px 0}img{width:100%;height:auto}figcaption{text-align:right}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font-size:15px;margin:18px 0}th,td{border-bottom:1px solid #dbe4e8;text-align:left;padding:10px 9px}th{background:#edf3f5;white-space:nowrap}.callout{background:#edf7f5;border-left:4px solid #258b81;padding:16px 20px}.formula{padding:18px 24px;background:#f3f6f8;border-radius:8px}summary{cursor:pointer;font-weight:600}details{padding:15px 0;border-top:1px solid #e0e7eb}li{margin:7px 0}.tag{font-size:13px;color:#667b86}@media(max-width:700px){main{padding:16px 10px}header,section{padding:23px 17px}.cards{grid-template-columns:1fr}h1{font-size:27px}h2{font-size:22px}table{font-size:13px}}@media print{body{background:#fff}main{padding:0}section{break-inside:avoid}nav,figcaption{display:none}}</style></head><body><main>'''
page+=f'''<header><div class="kicker">2026.10.04 분석 · SGIS 주민 인구 연결 완료</div><h1>폐의약품을 버리러 걷는 거리,<br>편의점 5곳을 더하면 얼마나 줄어들까?</h1><p class="lead">봉천동 일대와 창신·숭인동의 주민 위치를 보행망에 연결하고, 기존 수거망을 유지하면서 추가 편의점을 선정했습니다.</p><div class="cards">{''.join(cards)}</div><p>비교에 포함한 공개 인구는 총 <b>259,200명</b>입니다. 위 수치는 각 주민이 한 번 배출하러 간다고 가정했을 때의 인구 가중 평균 왕복거리입니다.</p><nav><a href="#maps">전후 지도</a><a href="#scale">추가 점포 수</a><a href="#trip">정류소 경유</a><a href="#age">고령층</a><a href="#sites">선정 후보</a><a href="#method">분석 방법</a></nav></header>'''
page+='''<section><h2>이번 분석의 질문</h2><p>기존 수거지가 있어도 찾아가는 데 긴 우회가 필요할 수 있습니다. <b>“수거지를 생활권에 더했을 때 주민의 배출 이동 부담을 얼마나 줄일 수 있는가?”</b>를 중심 질문으로 삼았습니다.</p><p>이를 <b>집에서 수거지까지의 왕복 배출</b>과 <b>정류소로 가면서 수거지를 들르는 경유 배출</b>로 나눴습니다. 주민 수·고령인구·보행망·정류소·수거 가능 품목은 각각 수요, 형평성, 이동 비용, 이동 상황, 이용 가능한 시설을 결정합니다.</p><div class="callout">선정 기준은 주변 시설의 개수를 합산한 점수가 아니라, <b>기존 수거망에 비해 줄어드는 주민의 보행거리</b>입니다. 기존 수거지와 효과가 겹치는 후보는 추가 이익이 작게 계산됩니다.</div></section>'''
page+=f'<section id="maps"><h2>1. 어느 곳의 이동 부담이 줄었나</h2><p>격자의 색이 옅어질수록 배출을 위한 왕복거리가 짧습니다. 왼쪽과 오른쪽에 같은 색 기준을 적용했고, 오른쪽 번호는 아래 선정 점포 표와 연결됩니다.</p>{maps}<p class="note">물약류 · 각 지역 5곳 · 전체 인구의 거리 최소화 결과. 100m 인구 격자 중심에서 계산. 회색은 인구 미공개 격자이며 0명으로 처리하지 않았습니다. 지도: SGIS, 스마트서울맵, 소상공인시장진흥공단, © OpenStreetMap contributors / ODbL.</p>{table(["지역","품목","기존 평균","5곳 추가 평균","감소율","상위 10% 경계 거리¹"],headline_rows)}<p class="note">¹ 인구 가중 90백분위 거리. 주민의 90%가 이 값 이하에 위치한다는 뜻입니다. 품목마다 5곳을 따로 최적화했으므로, 알약류와 물약류의 선정 점포가 모두 같지는 않습니다.</p>{img("05_동별_개선효과","동별 물약류 왕복거리 개선")}<p>지역 전체 평균과 함께 동별 개선을 확인할 수 있습니다. 지도에서 남은 짙은 격자는 편의점 5곳만으로 충분히 보완되지 않은 곳입니다.</p></section>'
page+=f'<section id="scale"><h2>2. 몇 곳을 추가하는 것이 좋은가</h2>{img("02_도입규모와_왕복거리","추가 점포 수에 따른 왕복거리")}<p>0·3·5·10·15곳을 각각 계산했습니다. 이 곡선으로 점포를 더 늘릴 때 얻는 추가 개선을 비교할 수 있습니다. 실제 도입 규모는 점포당 운영·회수 비용을 확인한 뒤 결정하는 것이 적절합니다.</p><p class="note">각 규모를 독립적으로 최적화했습니다. 5곳의 선정 점포가 10곳 선정안에 모두 포함된다는 뜻은 아닙니다. 고정 순서의 단계별 도입안은 별도 제약을 두어 계산해야 합니다.</p></section>'
page+=f'<section id="trip"><h2>3. 정류소로 가면서 배출할 때도 도움이 될까</h2><div class="formula"><b>추가 경유거리</b> = 집 → 수거지 → 정류소 거리 − 집 → 정류소 거리</div><p>각 거주지에서 보행거리상 가장 가까운 정류소를 하나 정했습니다. 아래 가운데 막대는 앞서 왕복거리로 선정한 <b>같은 5곳</b>을 활용한 결과이고, 오른쪽은 경유거리 감소를 목표로 다시 선정한 결과입니다.</p>{img("03_같은점포의_정류소경유효과","같은 점포의 정류소 경유 효과와 경유 최적화 비교")}<p>이 비교는 “배출만을 위한 방문”과 “이동 중 들르기”에 적합한 점포가 얼마나 겹치는지 보여줍니다. 정류소 위치만으로 실제 통근 경로나 배출 참여율을 알 수는 없어, 현재는 이동 상황을 가정한 비교로 사용합니다.</p></section>'
page+=f'<section id="age"><h2>4. 고령층을 우선하면 선정 결과가 달라질까</h2><p>전체 거리 감소를 최대로 만드는 배치를 먼저 찾고, 그 효과의 <b>95% 이상을 유지</b>하면서 65세 이상 주민의 거리를 가장 많이 줄이는 배치를 계산했습니다. 95%는 정책 선택을 보여주기 위한 조건이며, 90%·100%도 함께 비교했습니다.</p>{img("04_전체효과와_고령층개선","전체 효과와 고령층 개선의 절충")}{table(["지역","전체 평균: 효율안 → 고령층 우선안","65세 이상: 효율안 → 고령층 우선안","점포 구성"],fair_rows)}<p>창신·숭인동에서는 전체 평균거리가 약 11m 늘어나는 대신 고령층 평균거리를 약 27m 더 줄일 수 있었습니다. 봉천동에서는 고령층의 추가 개선이 약 6m로 상대적으로 작았습니다. <b>고령층 변수를 넣는 효과가 두 지역에서 같지는 않았습니다.</b></p><p class="note">고령인구는 집계구 대표점으로 계산했습니다. 미공개 연령값은 총인구와 공개 연령값을 이용해 하한·상한을 구했습니다. 그림·주 결과는 하한 가중치이며, 전체 인구의 100m 격자 결과와 절대 수준을 직접 비교해 연령 차이라고 해석하지 않습니다. {esc(" ".join(fair_notes))}</p></section>'
page+=f'<section id="sites"><h2>5. 물약류 왕복거리 기준 선정 후보</h2><p>지역별 5곳을 추가하는 효율안입니다. 점포명과 주소는 확보한 상가정보의 표기를 유지했습니다.</p>{sites}<p class="note">입지 분석 후보이며 점포의 참여·영업·보관 공간·실제 출입구를 확인한 목록은 아닙니다. 선정 전 확인 항목은 원본 자료의 표시를 그대로 남겼습니다.</p><h3>한 곳이 참여하지 않을 때의 대안</h3><p>선정 점포를 한 곳씩 제외하고, 나머지 4곳을 유지한 채 대체 후보를 찾았습니다. 아래는 각 지역에서 손실이 가장 큰 한 곳의 사례입니다.</p>{table(["지역","참여 불발 점포","평균거리 변화","대체 후보","대체 후"],risk_rows)}<p class="note">불참 확률을 추정한 값은 아니며, 5개 단일 불참 상황을 각각 계산한 결과입니다.</p></section>'
page+=f'<section><h2>6. 결과가 얼마나 안정적인가</h2><p>40개 기본 모형은 모두 설정한 허용 오차 안에서 최적해를 확인했습니다. 점포 수가 늘수록 평균거리가 증가하지 않는지, 전후 비교 인구가 같은지, 후보 지역이 주민 분석 범위와 맞는지도 검증했습니다.</p><details><summary>계단으로 표시된 길을 제외한 경우</summary>{table(["지역","공통 비교 인구","일반 보행망 전→후","계단 제외 전→후","경로 미연결 인구"],stairs_rows)}<p class="note">물약류, 같은 선정 5곳을 유지한 평가. 계단 제외 전후 모두 경로가 연결되고 지도 경계의 영향을 피하는 인구끼리 비교했습니다. 미연결 인구를 0거리로 넣지 않았습니다. OSM 계단 표기의 완전성은 확인되지 않았고 경사·승강기·문턱은 반영하지 않아 무장애 접근성을 뜻하지 않습니다.</p></details><details><summary>큰 인구 격자와 후보 출입 조건에 대한 점검</summary><p>가장 인구가 큰 격자를 한 개 제외하고 다시 선정하면 두 지역 모두 물약류 왕복거리 기준 5곳 중 4곳이 유지됐습니다. 봉천동의 해당 격자는 공개 인구 6,088명으로 분석 인구의 2.7%를 차지합니다. 이 격자는 기본 분석에 그대로 포함하고, 제외 결과를 별도 민감도로 남겼습니다.</p><p>시설 내부 출입 또는 층 정보 확인 표시가 있는 후보를 제외해도 두 지역의 물약류 왕복 5곳 선정안은 유지됐습니다. 이 확인만으로 다른 후보의 실제 영업·출입 상태가 보장되는 것은 아닙니다.</p></details><details><summary>350m 기준은 어떻게 다뤘나</summary><p>이번 최적화에는 350m 경계가 없습니다. 가까워지는 거리 자체를 계산하므로 349m와 351m가 전혀 다른 집단으로 나뉘지 않습니다. 200·350·500m 이내 보행 접근률은 이전 분석과 비교할 수 있는 보조 지표로만 별도 저장했습니다.</p></details></section>'
page+='''<section id="method"><h2>분석 범위와 계산 방법</h2><p><b>공간 범위:</b> 봉천동에 대응하는 9개 행정동, 창신·숭인동의 5개 행정동. 기존 수거지는 행정경계 밖의 인접 시설도 포함하며, 편의점 후보는 해당 행정동 범위 안에서 선정했습니다. 후보는 각각 142곳과 40곳입니다.</p><p><b>인구와 거리:</b> 2024년 100m 인구 격자를 2025년 2분기 행정경계와 연결했습니다. 도로망·시설은 2026년 자료이므로 “2024 인구 분포에 현재 공개 시설망을 적용한 분석”입니다. 봉천동 공개 인구 223,327명 중 223,164명, 창신·숭인동 36,036명 전원을 비교에 포함했습니다.</p><p><b>보행망:</b> OSM의 보행 금지·사유 접근 도로를 제외하고, 공유 노드를 통해 도로를 연결했습니다. 거주지와 시설을 가장 가까운 도로 구간에 연결하며 최대 연결 거리는 80m입니다. 연결 거리도 이동 비용에 포함했습니다. 지도 경계 밖을 경유하는 더 짧은 경로의 가능성은 거리 하한으로 점검했습니다.</p><p><b>수거 품목:</b> 알약류는 전용 수거함과 우체통, 물약류는 전용 수거함을 기존망으로 사용했습니다. 추가 편의점에는 해당 품목을 수거할 수 있는 설비를 도입하는 상황을 가정합니다. 시간대별 수거 가능 여부는 계산에 넣지 않았습니다.</p><div class="formula"><b>기본 모형</b><br>각 거주지의 이용 비용 = 기존 수거지와 선정 편의점 중 가장 짧은 이동거리<br>목표 = 모든 거주지의 [인구 × 이용 비용] 합 최소화<br>조건 = 추가 점포 수 ≤ K, 기존 수거망 유지<br><br><b>고령층 우선 모형</b><br>목표 = [65세 이상 인구 × 이용 비용] 합 최소화<br>추가 조건 = 전체 인구의 거리 감소가 효율안 감소량의 90%·95%·100% 이상</div><p>혼합정수 최적화로 계산했으며 점포 간 중복 혜택은 같은 주민에게 두 번 합산하지 않습니다. 실제 배출 횟수·회수량·사업비·영업시간 자료가 추가되면 수요와 운영 제약을 확장할 수 있습니다.</p><p class="note">SGIS 격자 인구에는 비밀보호 조정이 적용됩니다. 이번 결과는 공개 인구를 가중치로 사용한 잠재 보행거리이며, 관측된 개인 이동거리나 연간 총거리 절감량은 아닙니다. 연령 자료는 집계구 단위를 유지했습니다.</p></section>'''
page+=f'<section><h2>현재 결과에서 가져갈 이야기</h2><p><b>“기존 수거망을 유지하면서 생활권 거점을 더하면 배출을 위해 따로 걷는 거리와 이동 중 우회 부담을 줄일 수 있다.”</b> 주민 인구를 반영한 이번 결과가 이를 뒷받침합니다.</p><p>연구의 주축은 물약류의 왕복 이동 부담과 보완 입지로 두고, 정류소 경유와 고령층 우선 배치를 확장 분석으로 연결하면 흐름이 분명합니다. 물약류는 우체통을 이용할 수 없어 기존망과 추가 거점의 역할을 구분하기도 좋습니다.</p><p>다음 검증은 선정 후보 주변의 실제 횡단·출입구·계단 연결과 수거 운영 조건에 집중하면 됩니다. 경사에 따른 보행시간은 고도 자료를 연결한 뒤 별도로 계산해야 합니다.</p></section><section><h2>출처와 결과 자료</h2>{source_html}<p><a href="summary.json">전체 40개 기본 분석</a> · <a href="extensions.json">고령층·불참 대안·민감도 결과</a> · <a href="interpretation_metrics.json">동별·동일 점포 경유·거리별 접근률</a> · <a href="validation.json">결과 검증</a> · <a href="../../data/processed/source_manifest.json">자료 출처와 원본 해시</a></p><p class="note">자료: 각 제공기관 및 본 연구 계산. 모든 그림은 바로 아래 링크로 따로 저장할 수 있습니다. 분석·정리에는 AI의 코드 작성과 검토 지원을 사용했습니다. 이 페이지는 연구 검토용 결과물입니다.</p></section></main></body></html>'
path=R/'서울_주민기준_이동부담과_편의점최적화.html';path.write_text(page,encoding='utf-8')

# Extend provenance without rewriting or deleting earlier source records.
manifest_path=ROOT/'data/processed/source_manifest.json';manifest=read(manifest_path)
present={r.get('path') for r in manifest}
names=['SGIS 서울 2024 인구·연령 통계','SGIS 서울 2025Q2 행정동 경계','SGIS 서울 2025Q2 집계구 경계','SGIS 다사 격자 경계']
for rec,name in zip(INPUT['inputs'],names):
    rel='data/raw/'+rec['file']
    if rel not in present: manifest.append({'name':name,'path':rel,'sha256':rec['sha256'],'source_url':'https://sgis.mods.go.kr/view/pss/downloadList','obtained_on':'2026-10-04','reference_date':'2024 통계 / 2025 경계','note':'사용자 신청·승인 후 공식 다운로드. 신청자 개인정보는 분석 파일에 포함하지 않음.'})
for rec in read(ROOT/'data/raw/expanded_osm/manifest.json')['files']:
    rel=rec['file']
    if rel not in present: manifest.append({'name':'OSM 보행망 확장 추출','path':rel,'sha256':rec['sha256'],'source_url':rec.get('url',rec.get('source_url','https://api.openstreetmap.org/api/0.6/map')),'obtained_on':'2026-10-04','license':'ODbL / © OpenStreetMap contributors'})
dump(manifest_path,manifest)
dump(ROOT/'results/continuation_status.json',{'stage':'resident_models_completed','updated_on':'2026-10-04',
      'report':str(path),'models':40,'validations':'passed','scope':['봉천동 9개 행정동','창신·숭인동 5개 행정동'],
      'remaining_research_checks':['선정 점포 출입구·횡단·보행 연결 대조','고도 자료와 경사 이동시간','점포 운영·참여 조건'],
      'note':'수집·공간 연결·주민 가중 최적화 완료. 공모전 제출 또는 현장 검증 완료를 뜻하지 않음.'})
print(json.dumps({'report':str(path),'models_validated':len(checks),'figures':len(list(F.glob('*.png'))),'cross':cross},ensure_ascii=False,indent=2))


In [ ]:
import runpy
from IPython.display import display, Image

report_path = ROOT / 'src' / 'build_resident_report.py'
report_source = report_path.read_text(encoding='utf-8')


report_source = report_source.replace(
    "{esc(' '.join(fair_notes))}",
    '{esc(" ".join(fair_notes))}'
)

compile(report_source, str(report_path), 'exec')
report_path.write_text(report_source, encoding='utf-8')

run_stage(
    '06_검증과_그림생성',
    lambda: runpy.run_path(str(report_path), run_name='__main__')
)

validation = read_json('results/resident_analysis/validation.json')
assert validation['all_passed']
print(f"기본 결과 {validation['models_checked']}개 재집계 검증 통과")

def show_figure(name):
    display(Image(
        filename=str(ROOT / 'results/resident_analysis/figures' / name),
        width=1050
    ))

### 8.1 전체 도입 규모 비교

가로축은 추가 편의점 수, 세로축은 인구 가중 평균 왕복거리
이 그림의 5곳 지점을 기준으로 다음 지도와 확장 분석을 자세히 관찰

In [ ]:
show_figure('02_도입규모와_왕복거리.png')

### 8.2 편의점 5곳을 추가했을 때

물약류 왕복거리 기준입니다. 지도 번호는 같은 지역의 선정 점포 표 순서와 연결

In [ ]:
show_figure('01_bongcheon_왕복거리_전후지도.png')
show_figure('01_changsin_sungin_왕복거리_전후지도.png')
chosen = []
for r in summary:
    if r['medicine']=='liquid' and r['trip_type']=='roundtrip' and r['k']==5:
        for index, site in enumerate(r['selected_sites'], 1):
            chosen.append({'지역':area_names[r['area']], '지도번호':index, '후보':site['name'],
                           '주소':site['address'], '추가확인':', '.join(site['flags'])})
display(pd.DataFrame(chosen))

### 8.3 정류소로 가는 길에 들르는 효과


In [ ]:
show_figure('03_같은점포의_정류소경유효과.png')

### 8.4 전체 주민과 고령층의 개선 비교


In [ ]:
show_figure('04_전체효과와_고령층개선.png')

## 9. 보고서 결과와 비교


In [ ]:
reference = read_json('reference/report_summary.json')
key = lambda row: (row['area'], row['medicine'], row['trip_type'], row['k'])
ref_lookup = {key(r):r for r in reference}
result_checks = []
for r in summary:
    original = ref_lookup[key(r)]
    difference = abs(r['after_mean_m'] - original['after_mean_m'])
    result_checks.append({'지역':area_names[r['area']], '수거조건':r['medicine'], '이동':r['trip_type'],
                          '도입수':r['k'], '보고서와_평균거리차_m':difference,
                          '동일_인구':r['population']==original['population'],
                          '동일_점포조합':set(r['selected_indices'])==set(original['selected_indices'])})
comparison = pd.DataFrame(result_checks)
assert comparison['동일_인구'].all()
assert comparison['보고서와_평균거리차_m'].max() < .001
assert validation['all_passed']
notebook_validation = {
    'basic_models_checked':len(summary), 'same_population':bool(comparison['동일_인구'].all()),
    'maximum_mean_distance_difference_m':float(comparison['보고서와_평균거리차_m'].max()),
    'same_selected_combinations':int(comparison['동일_점포조합'].sum()),
    'model_enumeration_checks':model_check['enumeration_comparisons'],
    'input_files_hash_verified':len(checks), 'raw_inputs_rebuilt':REBUILD_INPUTS,
    'force_route_rebuild':REBUILD_ROUTES, 'all_passed':True,
}
(ROOT/'results/notebook/reproduction_validation.json').write_text(
    json.dumps(notebook_validation, ensure_ascii=False, indent=2), encoding='utf-8')
display(comparison.round(6))
print('전체 검증 완료. 최종 표·지도·그래프는 results/resident_analysis 폴더에 있습니다.')

## 10. 해석 범위와 자료 출처

- 계산값은 각 주민이 한 번 배출한다고 가정한 잠재 거리입니다. 실제 방문·배출량·연간 총절감량은 아닙니다.
- 고령인구는 집계구 대표점에 놓인다고 가정합니다. 주거 건물별 거주 분포와 대표점 위치 민감도는 반영하지 않았습니다.
- 점포의 실제 참여, 보관·회수 비용, 수거 가능 시간, 출입구·횡단 연결은 현장 확인 항목입니다.
- 계단 제외는 OSM 계단 태그 기준이며 경사나 모든 이동 장벽을 반영하지 않습니다.
- 350m는 선정 기준이 아닙니다. 코드는 거리 감소 자체를 최적화합니다.

원자료와 출처는 함께 제공한 `docs/DATA_SOURCES.md`, 분석 방법은 `docs/METHODS.md`를 참고하세요.
SGIS: https://sgis.mods.go.kr/view/pss/openDataIntrcn
상가정보: https://www.data.go.kr/data/15083033/fileData.do
수거함: https://map.seoul.go.kr/smgis2/
우체통: https://m.epost.go.kr/mobile/postFind/postMapSearch.jsp
정류소: https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do
도로망: © OpenStreetMap contributors / ODbL https://www.openstreetmap.org/copyright

분석·코드·시각화 작성에는 OpenAI Codex를 활용했습니다.
GitHub 공개용 묶음에는 원자료를 포함하지 않았습니다. 분석에 필요한 자료는 별도로 준비하며, 외부 재배포 시 제공기관의 이용 조건을 확인하세요.